In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

In [4]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [5]:
NOTEBOOK_DIR = Path.cwd().resolve()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

DAILY_DATA_DIR = PROJECT_ROOT / "data" / "daily"
MINUTE_DATA_DIR = PROJECT_ROOT / "data" / "minute"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
RESEARCH_DIR = PROJECT_ROOT / "research"

PROCESSED_DATA_DIR = OUTPUTS_DIR / "processed_data"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
RESEARCH_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook directory:", NOTEBOOK_DIR)
print("Project root:", PROJECT_ROOT)
print("Daily data directory:", DAILY_DATA_DIR)
print("Minute data directory:", MINUTE_DATA_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)

Notebook directory: /Users/kushagr/Desktop/astra-assignment/notebooks
Project root: /Users/kushagr/Desktop/astra-assignment
Daily data directory: /Users/kushagr/Desktop/astra-assignment/data/daily
Minute data directory: /Users/kushagr/Desktop/astra-assignment/data/minute
Processed data directory: /Users/kushagr/Desktop/astra-assignment/outputs/processed_data


In [6]:
assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"
assert DAILY_DATA_DIR.exists(), f"Daily data directory not found: {DAILY_DATA_DIR}"
assert MINUTE_DATA_DIR.exists(), f"Minute data directory not found: {MINUTE_DATA_DIR}"

print("All required directories were found.")

All required directories were found.


In [7]:
daily_files = sorted(DAILY_DATA_DIR.glob("*.parquet"))

print("Number of daily parquet files:", len(daily_files))

for file_path in daily_files[:10]:
    print(file_path.name)

Number of daily parquet files: 208
360ONE.parquet
ABB.parquet
ABCAPITAL.parquet
ADANIENSOL.parquet
ADANIENT.parquet
ADANIGREEN.parquet
ADANIPORTS.parquet
ADANIPOWER.parquet
ALKEM.parquet
AMBER.parquet


In [8]:
EXPECTED_SYMBOL_COUNT = 208

if len(daily_files) == EXPECTED_SYMBOL_COUNT:
    print("Daily file count matches the assignment.")
else:
    print(
        f"Warning: expected {EXPECTED_SYMBOL_COUNT} files, "
        f"but found {len(daily_files)}."
    )

Daily file count matches the assignment.


In [9]:
sample_daily_file = daily_files[0]
sample_symbol = sample_daily_file.stem

sample_daily_df = pd.read_parquet(sample_daily_file)

print("Sample file:", sample_daily_file.name)
print("Symbol:", sample_symbol)
print("Shape:", sample_daily_df.shape)

print(sample_daily_df.head())
print("\n" * 2)
print(sample_daily_df.info())


Sample file: 360ONE.parquet
Symbol: 360ONE
Shape: (1511, 6)
         date     open     high      low    close  volume
0  2020-06-01 209.1500 230.9500 209.1500 219.2500  185884
1  2020-06-02 224.9500 228.7500 218.9000 224.6500   22368
2  2020-06-03 228.7500 230.0000 221.2500 227.7000   34560
3  2020-06-04 233.7500 247.5000 228.7500 239.8500   77532
4  2020-06-05 244.0500 253.0500 236.5000 250.1500   22376



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1511 entries, 0 to 1510
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    1511 non-null   object 
 1   open    1511 non-null   float64
 2   high    1511 non-null   float64
 3   low     1511 non-null   float64
 4   close   1511 non-null   float64
 5   volume  1511 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 71.0+ KB
None


In [10]:
EXPECTED_DAILY_COLUMNS = [
    "date",
    "open",
    "high",
    "low",
    "close",
    "volume",
]

print("Columns found:", sample_daily_df.columns.tolist())

missing_columns = set(EXPECTED_DAILY_COLUMNS) - set(sample_daily_df.columns)

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("Expected daily columns are present.")

Columns found: ['date', 'open', 'high', 'low', 'close', 'volume']
Expected daily columns are present.


In [11]:
def load_daily_file(file_path: Path) -> pd.DataFrame:
    """
    Load one stock's daily parquet file and add its symbol.

    Parameters
    ----------
    file_path : Path
        Path to the parquet file.

    Returns
    -------
    pd.DataFrame
        Clean daily OHLCV data with a symbol column.
    """
    df = pd.read_parquet(file_path).copy()

    df.columns = [str(column).strip().lower() 
                  for column in df.columns
                  ]

    required_columns = {"date", "open", "high", "low", "close", "volume",}

    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{file_path.name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    df = df[["date","open","high","low","close","volume",]].copy()

    df["date"] = pd.to_datetime(df["date"], errors="raise",)

    df["symbol"] = file_path.stem

    return df

In [12]:
test_daily_df = load_daily_file(sample_daily_file)

print(test_daily_df.shape)
test_daily_df.head()

(1511, 7)


,date,open,high,low,close,volume,symbol
0,2020-06-01,209.1500,230.9500,209.1500,219.2500,185884,360ONE
1,2020-06-02,224.9500,228.7500,218.9000,224.6500,22368,360ONE
2,2020-06-03,228.7500,230.0000,221.2500,227.7000,34560,360ONE
3,2020-06-04,233.7500,247.5000,228.7500,239.8500,77532,360ONE
4,2020-06-05,244.0500,253.0500,236.5000,250.1500,22376,360ONE


In [13]:
daily_frames = []

for file_path in daily_files:
    stock_daily_df = load_daily_file(file_path)
    daily_frames.append(stock_daily_df)

daily_df = pd.concat(daily_frames,axis=0, ignore_index=True,)

daily_df = daily_df.sort_values(["symbol", "date"]).reset_index(drop=True)

daily_df = daily_df[["date", "symbol", "open", "high", "low", "close", "volume", ]]

print("Combined daily dataset shape:", daily_df.shape)
daily_df.head()

Combined daily dataset shape: (301483, 7)


,date,symbol,open,high,low,close,volume
0,2020-06-01,360ONE,209.1500,230.9500,209.1500,219.2500,185884
1,2020-06-02,360ONE,224.9500,228.7500,218.9000,224.6500,22368
2,2020-06-03,360ONE,228.7500,230.0000,221.2500,227.7000,34560
3,2020-06-04,360ONE,233.7500,247.5000,228.7500,239.8500,77532
4,2020-06-05,360ONE,244.0500,253.0500,236.5000,250.1500,22376


In [14]:
print("Rows:", len(daily_df))
print("Symbols:", daily_df["symbol"].nunique())
print("Trading dates:", daily_df["date"].nunique())
print("Date range:", daily_df["date"].min(), "to", daily_df["date"].max(),)

Rows: 301483
Symbols: 208
Trading dates: 1511
Date range: 2020-06-01 00:00:00 to 2026-06-30 00:00:00


In [15]:
duplicate_mask = daily_df.duplicated(subset=["symbol", "date"],keep=False,)

duplicate_count = int(duplicate_mask.sum())

print("Duplicate symbol-date rows:", duplicate_count)

if duplicate_count > 0:
    display(
        daily_df.loc[duplicate_mask]
        .sort_values(["symbol", "date"])
        .head(20)
    )

Duplicate symbol-date rows: 0


In [16]:
missing_summary = daily_df.isna().sum().to_frame(name="missing_count")
missing_summary["missing_pct"] = (missing_summary["missing_count"]/ len(daily_df)* 100)
missing_summary

,missing_count,missing_pct
date,0,0.0000
symbol,0,0.0000
open,0,0.0000
high,0,0.0000
low,0,0.0000
close,0,0.0000
volume,0,0.0000


In [17]:
quality_checks = pd.Series(
    {
        "non_positive_open": (daily_df["open"] <= 0).sum(),
        "non_positive_high": (daily_df["high"] <= 0).sum(),
        "non_positive_low": (daily_df["low"] <= 0).sum(),
        "non_positive_close": (daily_df["close"] <= 0).sum(),
        "negative_volume": (daily_df["volume"] < 0).sum(),
        "high_below_low": (
            daily_df["high"] < daily_df["low"]
        ).sum(),
        "high_below_open": (
            daily_df["high"] < daily_df["open"]
        ).sum(),
        "high_below_close": (
            daily_df["high"] < daily_df["close"]
        ).sum(),
        "low_above_open": (
            daily_df["low"] > daily_df["open"]
        ).sum(),
        "low_above_close": (
            daily_df["low"] > daily_df["close"]
        ).sum(),
    },
    name="count",
)

quality_checks

non_positive_open     0
non_positive_high     0
non_positive_low      0
non_positive_close    0
negative_volume       0
high_below_low        0
high_below_open       0
high_below_close      0
low_above_open        0
low_above_close       0
Name: count, dtype: int64

In [18]:
rows_per_symbol = (daily_df.groupby("symbol").size().sort_values())

rows_per_symbol.describe()

count     208.0000
mean    1,449.4375
std       210.6202
min       378.0000
25%     1,511.0000
50%     1,511.0000
75%     1,511.0000
max     1,511.0000
dtype: float64

In [19]:
rows_per_symbol.head(20)

symbol
VMM            378
SWIGGY         401
WAAREEENER     413
HYUNDAI        417
PREMIERENE     451
IREDA          640
JIOFIN         708
MANKIND        780
KFINTECH       866
KAYNES         893
DELHIVERY     1017
LICI          1022
PAYTM         1143
POLICYBZR     1146
NYKAA         1149
ETERNAL       1223
SONACOMS      1243
LODHA         1289
KALYANKJIL    1302
IRFC          1341
dtype: int64

In [20]:
symbols_per_date = (daily_df.groupby("date")["symbol"].nunique().sort_values())

symbols_per_date.head(10)

date
2020-06-01    184
2020-07-15    184
2020-07-16    184
2020-07-17    184
2020-07-20    184
2020-07-22    184
2020-07-23    184
2020-07-24    184
2020-07-27    184
2020-07-28    184
Name: symbol, dtype: int64

In [21]:
daily_panel_path = (PROCESSED_DATA_DIR / "daily_panel_raw.parquet")

daily_df.to_parquet(daily_panel_path, index=False,)

print("Saved combined daily panel to:")
print(daily_panel_path)

Saved combined daily panel to:
/Users/kushagr/Desktop/astra-assignment/outputs/processed_data/daily_panel_raw.parquet


In [22]:
daily_df.head()

,date,symbol,open,high,low,close,volume
0,2020-06-01,360ONE,209.1500,230.9500,209.1500,219.2500,185884
1,2020-06-02,360ONE,224.9500,228.7500,218.9000,224.6500,22368
2,2020-06-03,360ONE,228.7500,230.0000,221.2500,227.7000,34560
3,2020-06-04,360ONE,233.7500,247.5000,228.7500,239.8500,77532
4,2020-06-05,360ONE,244.0500,253.0500,236.5000,250.1500,22376


# Target Construction


In [23]:
target_df = daily_df.copy()

target_df = target_df.sort_values(["symbol", "date"]).reset_index(drop=True)

print(target_df.shape)
target_df.head()

(301483, 7)


,date,symbol,open,high,low,close,volume
0,2020-06-01,360ONE,209.1500,230.9500,209.1500,219.2500,185884
1,2020-06-02,360ONE,224.9500,228.7500,218.9000,224.6500,22368
2,2020-06-03,360ONE,228.7500,230.0000,221.2500,227.7000,34560
3,2020-06-04,360ONE,233.7500,247.5000,228.7500,239.8500,77532
4,2020-06-05,360ONE,244.0500,253.0500,236.5000,250.1500,22376


In [24]:
symbol_groups = target_df.groupby("symbol",sort=False,)

target_df["target_date"] = (symbol_groups["date"].shift(-1))

target_df["next_open"] = (symbol_groups["open"].shift(-1))

In [25]:
target_df["actual_return_pct"] = ((target_df["next_open"]/ target_df["close"])- 1) * 100

target_df["actual_direction"] = np.where(target_df["actual_return_pct"] >= 0,1,-1,)

target_df["actual_magnitude_pct"] = (target_df["actual_return_pct"].abs())

In [26]:
target_df["calendar_gap_days"] = (target_df["target_date"]- target_df["date"]).dt.days

In [27]:
target_columns = ["date","target_date","symbol","close","next_open","actual_return_pct","actual_direction","actual_magnitude_pct","calendar_gap_days",]

target_df[target_columns].head(5)

,date,target_date,symbol,close,next_open,actual_return_pct,actual_direction,actual_magnitude_pct,calendar_gap_days
0,2020-06-01,2020-06-02,360ONE,219.2500,224.9500,2.5998,1,2.5998,1.0000
1,2020-06-02,2020-06-03,360ONE,224.6500,228.7500,1.8251,1,1.8251,1.0000
2,2020-06-03,2020-06-04,360ONE,227.7000,233.7500,2.6570,1,2.6570,1.0000
3,2020-06-04,2020-06-05,360ONE,239.8500,244.0500,1.7511,1,1.7511,1.0000
4,2020-06-05,2020-06-08,360ONE,250.1500,251.7500,0.6396,1,0.6396,3.0000


In [28]:
sample_symbol = "RELIANCE"

target_df.loc[target_df["symbol"] == sample_symbol, target_columns,].head(5)

,date,target_date,symbol,close,next_open,actual_return_pct,actual_direction,actual_magnitude_pct,calendar_gap_days
243141,2020-06-01,2020-06-02,RELIANCE,724.6000,727.3000,0.3726,1,0.3726,1.0000
243142,2020-06-02,2020-06-03,RELIANCE,731.9000,736.3500,0.6080,1,0.6080,1.0000
243143,2020-06-03,2020-06-04,RELIANCE,734.7500,735.9000,0.1565,1,0.1565,1.0000
243144,2020-06-04,2020-06-05,RELIANCE,752.9000,760.2000,0.9696,1,0.9696,1.0000
243145,2020-06-05,2020-06-08,RELIANCE,753.8500,771.3000,2.3148,1,2.3148,3.0000


In [29]:
valid_target_mask = target_df["target_date"].notna()

invalid_target_dates = (target_df.loc[valid_target_mask, "target_date"] <= target_df.loc[valid_target_mask, "date"]).sum()

print("Rows where target_date is not after date:",invalid_target_dates,)

Rows where target_date is not after date: 0


In [30]:
target_df["actual_direction"].value_counts(dropna=False)

actual_direction
 1    214039
-1     87444
Name: count, dtype: int64

In [31]:
missing_target_mask = target_df["actual_return_pct"].isna()

target_df.loc[missing_target_mask,"actual_direction",] = np.nan

In [32]:
target_df["actual_direction"].value_counts(dropna=False)

actual_direction
1.0000     214039
-1.0000     87236
NaN           208
Name: count, dtype: int64

In [33]:
negative_magnitude_count = (target_df["actual_magnitude_pct"] < 0).sum()

print("Negative magnitude rows:",negative_magnitude_count,)

Negative magnitude rows: 0


In [34]:
target_df["calendar_gap_days"].value_counts(dropna=False).sort_index()

calendar_gap_days
1.0000      228739
2.0000       10085
3.0000       55846
4.0000        6408
5.0000         196
112.0000         1
NaN            208
Name: count, dtype: int64

In [35]:
missing_target_summary = target_df[["target_date","next_open","actual_return_pct","actual_direction","actual_magnitude_pct",]].isna().sum()

missing_target_summary

target_date             208
next_open               208
actual_return_pct       208
actual_direction        208
actual_magnitude_pct    208
dtype: int64

In [36]:
daily_with_targets_full = target_df.copy()


daily_with_targets = (
    target_df
    .dropna(
        subset=[
            "target_date",
            "next_open",
            "actual_return_pct",
            "actual_direction",
            "actual_magnitude_pct",
        ]
    )
    .copy()
)

daily_with_targets["actual_direction"] = (daily_with_targets["actual_direction"].astype("int8"))

print("Rows before dropping missing targets:",len(target_df),)

print("Rows after dropping missing targets:",len(daily_with_targets),)

print("Rows removed:",len(target_df) - len(daily_with_targets),)

Rows before dropping missing targets: 301483
Rows after dropping missing targets: 301275
Rows removed: 208


In [37]:
daily_with_targets = daily_with_targets.rename(columns={"date": "pred_date"})


daily_with_targets[["pred_date","target_date","symbol","close","next_open","actual_return_pct","actual_direction","actual_magnitude_pct","calendar_gap_days",]].head()

,pred_date,target_date,symbol,close,next_open,actual_return_pct,actual_direction,actual_magnitude_pct,calendar_gap_days
0,2020-06-01,2020-06-02,360ONE,219.2500,224.9500,2.5998,1,2.5998,1.0000
1,2020-06-02,2020-06-03,360ONE,224.6500,228.7500,1.8251,1,1.8251,1.0000
2,2020-06-03,2020-06-04,360ONE,227.7000,233.7500,2.6570,1,2.6570,1.0000
3,2020-06-04,2020-06-05,360ONE,239.8500,244.0500,1.7511,1,1.7511,1.0000
4,2020-06-05,2020-06-08,360ONE,250.1500,251.7500,0.6396,1,0.6396,3.0000


In [38]:
assert not daily_with_targets[["pred_date","target_date","symbol","actual_return_pct","actual_direction","actual_magnitude_pct",]].isna().any().any()

assert (daily_with_targets["target_date"] > daily_with_targets["pred_date"]).all()

assert daily_with_targets["actual_direction"].isin([-1, 1]).all()

assert (daily_with_targets["actual_magnitude_pct"] >= 0).all()

assert np.allclose(daily_with_targets["actual_magnitude_pct"],daily_with_targets["actual_return_pct"].abs(),)

print("All target-construction checks passed.")

All target-construction checks passed.


In [39]:
target_panel_path = (PROCESSED_DATA_DIR/ "daily_panel_with_targets_v1.parquet")

daily_with_targets.to_parquet(target_panel_path,index=False,)

print("Saved target panel to:", target_panel_path)

Saved target panel to: /Users/kushagr/Desktop/astra-assignment/outputs/processed_data/daily_panel_with_targets_v1.parquet


In [40]:
daily_with_targets.head()

,pred_date,symbol,open,high,low,close,volume,target_date,next_open,actual_return_pct,actual_direction,actual_magnitude_pct,calendar_gap_days
0,2020-06-01,360ONE,209.1500,230.9500,209.1500,219.2500,185884,2020-06-02,224.9500,2.5998,1,2.5998,1.0000
1,2020-06-02,360ONE,224.9500,228.7500,218.9000,224.6500,22368,2020-06-03,228.7500,1.8251,1,1.8251,1.0000
2,2020-06-03,360ONE,228.7500,230.0000,221.2500,227.7000,34560,2020-06-04,233.7500,2.6570,1,2.6570,1.0000
3,2020-06-04,360ONE,233.7500,247.5000,228.7500,239.8500,77532,2020-06-05,244.0500,1.7511,1,1.7511,1.0000
4,2020-06-05,360ONE,244.0500,253.0500,236.5000,250.1500,22376,2020-06-08,251.7500,0.6396,1,0.6396,3.0000




### Daily Feature Engineering — Version 1

Daily features are calculated separately within each stock.

All features for prediction date `T` use only information available by the close of `T`.

Historical overnight features are shifted by one row because `actual_return_pct`
for row `T` is only realised at the open of `target_date`.

In [41]:
daily_features_df = daily_with_targets.copy()

daily_features_df = daily_features_df.sort_values(["symbol", "pred_date"]).reset_index(drop=True)

print("Starting shape:", daily_features_df.shape)
daily_features_df.head()

Starting shape: (301275, 13)


,pred_date,symbol,open,high,low,close,volume,target_date,next_open,actual_return_pct,actual_direction,actual_magnitude_pct,calendar_gap_days
0,2020-06-01,360ONE,209.1500,230.9500,209.1500,219.2500,185884,2020-06-02,224.9500,2.5998,1,2.5998,1.0000
1,2020-06-02,360ONE,224.9500,228.7500,218.9000,224.6500,22368,2020-06-03,228.7500,1.8251,1,1.8251,1.0000
2,2020-06-03,360ONE,228.7500,230.0000,221.2500,227.7000,34560,2020-06-04,233.7500,2.6570,1,2.6570,1.0000
3,2020-06-04,360ONE,233.7500,247.5000,228.7500,239.8500,77532,2020-06-05,244.0500,1.7511,1,1.7511,1.0000
4,2020-06-05,360ONE,244.0500,253.0500,236.5000,250.1500,22376,2020-06-08,251.7500,0.6396,1,0.6396,3.0000


In [42]:
symbol_groups = daily_features_df.groupby("symbol",sort=False,group_keys=False,)

In [43]:
# The current row's actual_return_pct belongs to the future target.
# Shift by one so the feature contains only previously realised gaps.

daily_features_df["lagged_overnight_return_1d"] = (symbol_groups["actual_return_pct"].shift(1))

daily_features_df["gap_mean_20d"] = (daily_features_df.groupby("symbol")["lagged_overnight_return_1d"].transform(lambda series: series.rolling(window=20,min_periods=20,).mean()))

daily_features_df["overnight_std_20d"] = (daily_features_df.groupby("symbol")["lagged_overnight_return_1d"].transform(lambda series: series.rolling(window=20,min_periods=20,).std()))

daily_features_df["gap_positive_fraction_20d"] = (daily_features_df.groupby("symbol")["lagged_overnight_return_1d"].transform(lambda series: (series.ge(0).where(series.notna()).rolling(window=20,min_periods=20,).mean())))

In [44]:
overnight_feature_columns = ["pred_date","target_date","symbol","actual_return_pct","lagged_overnight_return_1d","gap_mean_20d","overnight_std_20d","gap_positive_fraction_20d",]

daily_features_df.loc[daily_features_df["symbol"] == "RELIANCE",overnight_feature_columns,].head(25)

,pred_date,target_date,symbol,actual_return_pct,lagged_overnight_return_1d,gap_mean_20d,overnight_std_20d,gap_positive_fraction_20d
242974,2020-06-01,2020-06-02,RELIANCE,0.3726,NaN,NaN,NaN,NaN
242975,2020-06-02,2020-06-03,RELIANCE,0.6080,0.3726,NaN,NaN,NaN
242976,2020-06-03,2020-06-04,RELIANCE,0.1565,0.6080,NaN,NaN,NaN
242977,2020-06-04,2020-06-05,RELIANCE,0.9696,0.1565,NaN,NaN,NaN
242978,2020-06-05,2020-06-08,RELIANCE,2.3148,0.9696,NaN,NaN,NaN
242979,2020-06-08,2020-06-09,RELIANCE,-0.5816,2.3148,NaN,NaN,NaN
242980,2020-06-09,2020-06-10,RELIANCE,0.3890,-0.5816,NaN,NaN,NaN
242981,2020-06-10,2020-06-11,RELIANCE,-0.3937,0.3890,NaN,NaN,NaN
242982,2020-06-11,2020-06-12,RELIANCE,-2.4560,-0.3937,NaN,NaN,NaN
242983,2020-06-12,2020-06-15,RELIANCE,-1.4923,-2.4560,NaN,NaN,NaN


In [45]:
daily_features_df["return_1d"] = (symbol_groups["close"].pct_change(1) * 100)

daily_features_df["return_5d"] = (symbol_groups["close"].pct_change(5) * 100)

daily_features_df["return_20d"] = (symbol_groups["close"].pct_change(20) * 100)

In [46]:
daily_features_df["daily_volatility_5d"] = (daily_features_df.groupby("symbol")["return_1d"].transform(lambda series: series.rolling(window=5,min_periods=5,).std()))

daily_features_df["daily_volatility_20d"] = (daily_features_df.groupby("symbol")["return_1d"].transform(lambda series: series.rolling(window=20,min_periods=20,).std()))

In [47]:
daily_features_df["volume_mean_20d_lagged"] = (symbol_groups["volume"].transform(lambda series: (series.shift(1).rolling(window=20,min_periods=20,).mean())))

daily_features_df["volume_std_20d_lagged"] = (symbol_groups["volume"].transform(lambda series: (series.shift(1).rolling(window=20,min_periods=20,).std())))

daily_features_df["volume_zscore_20d"] = ((daily_features_df["volume"]- daily_features_df["volume_mean_20d_lagged"])/ daily_features_df["volume_std_20d_lagged"])

In [48]:
daily_features_df["volume_mean_5d"] = (symbol_groups["volume"].transform(lambda series: series.rolling(window=5,min_periods=5,).mean()))

daily_features_df["volume_mean_20d"] = (symbol_groups["volume"].transform(lambda series: series.rolling(window=20,min_periods=20,).mean()))

daily_features_df["volume_trend_5d_20d"] = (daily_features_df["volume_mean_5d"] / daily_features_df["volume_mean_20d"])

In [49]:
daily_features_df["day_of_week"] = (daily_features_df["pred_date"].dt.dayofweek)

In [50]:
daily_feature_columns = ["lagged_overnight_return_1d","return_1d","return_5d","return_20d","daily_volatility_20d","overnight_std_20d","daily_volatility_5d","gap_mean_20d","gap_positive_fraction_20d","volume_zscore_20d","volume_trend_5d_20d","calendar_gap_days","day_of_week",]

print("Number of unique Version 1 daily features:", len(daily_feature_columns))

Number of unique Version 1 daily features: 13


In [51]:
daily_feature_missingness = daily_features_df[daily_feature_columns].isna().sum().to_frame("missing_count")

daily_feature_missingness["missing_pct"] = (daily_feature_missingness["missing_count"] / len(daily_features_df) * 100)

daily_feature_missingness.sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
return_20d,4160,1.3808
daily_volatility_20d,4160,1.3808
overnight_std_20d,4160,1.3808
gap_mean_20d,4160,1.3808
gap_positive_fraction_20d,4160,1.3808
volume_zscore_20d,4160,1.3808
volume_trend_5d_20d,3952,1.3118
return_5d,1040,0.3452
daily_volatility_5d,1040,0.3452
lagged_overnight_return_1d,208,0.0690


In [52]:
daily_feature_infinite_counts = pd.Series({column: np.isinf(daily_features_df[column]).sum() for column in daily_feature_columns},name="infinite_count",)

daily_feature_infinite_counts

lagged_overnight_return_1d    0
return_1d                     0
return_5d                     0
return_20d                    0
daily_volatility_20d          0
overnight_std_20d             0
daily_volatility_5d           0
gap_mean_20d                  0
gap_positive_fraction_20d     0
volume_zscore_20d             0
volume_trend_5d_20d           0
calendar_gap_days             0
day_of_week                   0
Name: infinite_count, dtype: int64

In [53]:
daily_features_df[daily_feature_columns] = (daily_features_df[daily_feature_columns].replace([np.inf, -np.inf], np.nan))

In [54]:
assert daily_features_df["day_of_week"].between(0, 6).all()

assert (daily_features_df["calendar_gap_days"] > 0).all()

valid_positive_fraction = daily_features_df["gap_positive_fraction_20d"].dropna()

assert valid_positive_fraction.between(0, 1).all()

print("Daily feature sanity checks passed.")

Daily feature sanity checks passed.


In [55]:
sample_columns = ["pred_date","symbol","lagged_overnight_return_1d","gap_mean_20d","overnight_std_20d","gap_positive_fraction_20d","return_1d","return_5d","return_20d","daily_volatility_5d","daily_volatility_20d","volume_zscore_20d","volume_trend_5d_20d","calendar_gap_days","day_of_week","actual_return_pct",]

daily_features_df.loc[daily_features_df["symbol"] == "RELIANCE",sample_columns,].tail(10)

,pred_date,symbol,lagged_overnight_return_1d,gap_mean_20d,overnight_std_20d,gap_positive_fraction_20d,return_1d,return_5d,return_20d,daily_volatility_5d,daily_volatility_20d,volume_zscore_20d,volume_trend_5d_20d,calendar_gap_days,day_of_week,actual_return_pct
244474,2026-06-15,RELIANCE,1.7247,0.1574,0.7391,0.6000,1.0828,3.4592,-2.1999,1.1672,1.2503,0.1440,1.0137,1.0000,0,0.4897
244475,2026-06-16,RELIANCE,0.4897,0.1909,0.7382,0.6500,1.6679,4.6959,-0.5315,1.2322,1.3117,0.1376,0.9403,1.0000,1,0.3161
244476,2026-06-17,RELIANCE,0.3161,0.1962,0.7387,0.6500,0.2935,5.8707,0.7560,0.8908,1.2930,-0.9885,0.8440,1.0000,2,-0.2026
244477,2026-06-18,RELIANCE,-0.2026,0.2012,0.7355,0.6500,-0.3452,5.1544,-2.3240,1.0784,1.1205,-0.2070,0.8661,1.0000,3,-0.0075
244478,2026-06-19,RELIANCE,-0.0075,0.1732,0.7321,0.6000,-1.4005,1.2761,-2.9713,1.2025,1.1493,0.9741,0.9934,3.0000,4,0.5498
244479,2026-06-22,RELIANCE,0.5498,0.1992,0.7359,0.6000,1.2982,1.4920,-2.0672,1.2425,1.1893,-0.5845,0.9136,1.0000,0,0.1809
244480,2026-06-23,RELIANCE,0.1809,0.1629,0.7168,0.6000,-1.2816,-1.4524,-4.2063,1.1272,1.1919,-0.3257,0.8582,1.0000,1,-0.2902
244481,2026-06-24,RELIANCE,-0.2902,0.1736,0.7078,0.6000,0.3131,-1.4332,-3.1483,1.1298,1.1893,-1.0020,0.8756,1.0000,2,0.3350
244482,2026-06-25,RELIANCE,0.3350,0.2025,0.7017,0.6500,0.3426,-0.7530,-2.3991,1.1617,1.1924,-0.7439,0.8461,4.0000,3,-0.7663
244483,2026-06-29,RELIANCE,-0.7663,0.1142,0.7072,0.6000,-1.2973,-0.6491,-1.5289,1.1345,1.1275,-0.5955,0.7861,1.0000,0,0.4535


In [56]:
daily_features_df[daily_feature_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
lagged_overnight_return_1d,"301,067.0000",0.2144,1.0074,-34.7406,-0.0672,0.1934,0.5803,63.8095
return_1d,"301,067.0000",0.1246,2.2480,-32.9341,-1.0334,0.0303,1.1607,85.7143
return_5d,"300,235.0000",0.6247,5.1696,-53.8122,-2.2604,0.3411,3.1310,94.9206
return_20d,"297,115.0000",2.4730,10.6418,-72.4351,-3.9055,1.5391,7.6362,139.8804
daily_volatility_20d,"297,115.0000",2.0491,0.9053,0.1686,1.4255,1.8751,2.4575,19.2841
overnight_std_20d,"297,115.0000",0.8305,0.5299,0.0969,0.4893,0.6932,1.0070,14.2957
daily_volatility_5d,"300,235.0000",1.8859,1.2019,0.0004,1.0850,1.6113,2.3709,38.8828
gap_mean_20d,"297,115.0000",0.2130,0.2810,-4.8037,0.0592,0.1974,0.3492,4.5780
gap_positive_fraction_20d,"297,115.0000",0.7103,0.1325,0.0500,0.6000,0.7000,0.8000,1.0000
volume_zscore_20d,"297,115.0000",0.1523,2.2482,-4.6068,-0.6732,-0.3008,0.3350,201.7907


In [57]:
feature_correlation = daily_features_df[daily_feature_columns].corr()

feature_correlation

,lagged_overnight_return_1d,return_1d,return_5d,return_20d,daily_volatility_20d,overnight_std_20d,daily_volatility_5d,gap_mean_20d,gap_positive_fraction_20d,volume_zscore_20d,volume_trend_5d_20d,calendar_gap_days,day_of_week
lagged_overnight_return_1d,1.0000,0.3678,0.2194,0.1247,0.0728,0.0305,0.0747,0.2593,0.1612,0.0341,0.0536,-0.0158,-0.0058
return_1d,0.3678,1.0000,0.4461,0.2278,0.0450,0.0167,0.0825,0.0823,0.0459,0.1518,0.0741,-0.0186,-0.0088
return_5d,0.2194,0.4461,1.0000,0.5048,0.0982,0.0316,0.1541,0.2102,0.1118,0.1019,0.2399,-0.0058,0.0059
return_20d,0.1247,0.2278,0.5048,1.0000,0.2060,0.0597,0.1347,0.4438,0.2361,0.0270,0.0750,-0.0034,0.0070
daily_volatility_20d,0.0728,0.0450,0.0982,0.2060,1.0000,0.6262,0.6298,0.2531,0.0206,0.0194,0.0316,-0.0005,-0.0051
overnight_std_20d,0.0305,0.0167,0.0316,0.0597,0.6262,1.0000,0.3603,0.0856,-0.1663,0.0048,-0.0019,0.0042,-0.0136
daily_volatility_5d,0.0747,0.0825,0.1541,0.1347,0.6298,0.3603,1.0000,0.1340,0.0085,0.1656,0.4322,0.0021,-0.0030
gap_mean_20d,0.2593,0.0823,0.2102,0.4438,0.2531,0.0856,0.1340,1.0000,0.6261,-0.0029,0.0079,-0.0076,0.0108
gap_positive_fraction_20d,0.1612,0.0459,0.1118,0.2361,0.0206,-0.1663,0.0085,0.6261,1.0000,0.0073,0.0141,-0.0117,0.0105
volume_zscore_20d,0.0341,0.1518,0.1019,0.0270,0.0194,0.0048,0.1656,-0.0029,0.0073,1.0000,0.3846,0.0140,0.0203


In [58]:
usable_rows = daily_features_df.dropna(subset=daily_feature_columns)

print("Total rows:", len(daily_features_df))
print("Rows with all daily features available:", len(usable_rows))
print("Coverage: {:.2f}%".format(len(usable_rows) / len(daily_features_df) * 100))

Total rows: 301275
Rows with all daily features available: 297115
Coverage: 98.62%


In [59]:
pd.DataFrame(
{
        "missing": daily_features_df[daily_feature_columns].isna().sum(),
        "missing_pct": daily_features_df[daily_feature_columns].isna().mean() * 100,
    }
).sort_values("missing_pct", ascending=False)

,missing,missing_pct
return_20d,4160,1.3808
daily_volatility_20d,4160,1.3808
overnight_std_20d,4160,1.3808
gap_mean_20d,4160,1.3808
gap_positive_fraction_20d,4160,1.3808
volume_zscore_20d,4160,1.3808
volume_trend_5d_20d,3952,1.3118
return_5d,1040,0.3452
daily_volatility_5d,1040,0.3452
lagged_overnight_return_1d,208,0.0690


In [60]:
daily_features_path = PROCESSED_DATA_DIR / "daily_features_v1.parquet"

daily_features_df.to_parquet(daily_features_path,index=False,)

print("Saved daily feature panel to:")
print(daily_features_path)

Saved daily feature panel to:
/Users/kushagr/Desktop/astra-assignment/outputs/processed_data/daily_features_v1.parquet


### Minute Feature Engineering

In [61]:
minute_files = sorted(MINUTE_DATA_DIR.glob("*.parquet"))

print("Number of minute parquet files:", len(minute_files))

for file_path in minute_files[:5]: 
    print(file_path.name)

Number of minute parquet files: 208
360ONE.parquet
ABB.parquet
ABCAPITAL.parquet
ADANIENSOL.parquet
ADANIENT.parquet


In [62]:
daily_symbols = {file_path.stem for file_path in daily_files}
minute_symbols = {file_path.stem for file_path in minute_files}

print("Daily symbols:", len(daily_symbols))
print("Minute symbols:", len(minute_symbols))
print("Present only in daily:", sorted(daily_symbols - minute_symbols))
print("Present only in minute:", sorted(minute_symbols - daily_symbols))

Daily symbols: 208
Minute symbols: 208
Present only in daily: []
Present only in minute: []


In [63]:
sample_minute_file = MINUTE_DATA_DIR / "RELIANCE.parquet"

if not sample_minute_file.exists():
    sample_minute_file = minute_files[0]

sample_minute_symbol = sample_minute_file.stem

print("Sample file:", sample_minute_file.name)
print("Sample symbol:", sample_minute_symbol)

Sample file: RELIANCE.parquet
Sample symbol: RELIANCE


In [64]:
sample_minute_df = pd.read_parquet(sample_minute_file).copy()

print("Shape:", sample_minute_df.shape)

sample_minute_df.head()

Shape: (563457, 6)


,timestamp,open,high,low,close,volume
0,2020-06-01 09:15:00,705.4000,706.0500,703.4500,704.7000,772062
1,2020-06-01 09:16:00,704.7000,705.6000,703.8000,703.9000,361144
2,2020-06-01 09:17:00,703.8000,706.3000,703.7000,706.3000,310700
3,2020-06-01 09:18:00,706.8000,708.6500,706.0000,708.1000,265712
4,2020-06-01 09:19:00,708.5000,709.9000,707.0000,708.8000,256512


In [65]:
print("Columns:", sample_minute_df.columns.tolist())

sample_minute_df.info()

Columns: ['timestamp', 'open', 'high', 'low', 'close', 'volume']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 563457 entries, 0 to 563456
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   timestamp  563457 non-null  datetime64[ns]
 1   open       563457 non-null  float64       
 2   high       563457 non-null  float64       
 3   low        563457 non-null  float64       
 4   close      563457 non-null  float64       
 5   volume     563457 non-null  int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 25.8 MB


In [66]:
sample_minute_df.columns = [str(column).strip().lower() for column in sample_minute_df.columns]

sample_minute_df["timestamp"] = pd.to_datetime(sample_minute_df["timestamp"], errors="raise")

sample_minute_df = sample_minute_df.sort_values("timestamp").reset_index(drop=True)

sample_minute_df["date"] = sample_minute_df["timestamp"].dt.normalize()
sample_minute_df["time"] = sample_minute_df["timestamp"].dt.time
sample_minute_df["symbol"] = sample_minute_symbol

sample_minute_df.head()

,timestamp,open,high,low,close,volume,date,time,symbol
0,2020-06-01 09:15:00,705.4000,706.0500,703.4500,704.7000,772062,2020-06-01,09:15:00,RELIANCE
1,2020-06-01 09:16:00,704.7000,705.6000,703.8000,703.9000,361144,2020-06-01,09:16:00,RELIANCE
2,2020-06-01 09:17:00,703.8000,706.3000,703.7000,706.3000,310700,2020-06-01,09:17:00,RELIANCE
3,2020-06-01 09:18:00,706.8000,708.6500,706.0000,708.1000,265712,2020-06-01,09:18:00,RELIANCE
4,2020-06-01 09:19:00,708.5000,709.9000,707.0000,708.8000,256512,2020-06-01,09:19:00,RELIANCE


In [67]:
print("First timestamp:", sample_minute_df["timestamp"].min())
print("Last timestamp:", sample_minute_df["timestamp"].max())
print("Number of trading dates:", sample_minute_df["date"].nunique())

First timestamp: 2020-06-01 09:15:00
Last timestamp: 2026-06-29 15:29:00
Number of trading dates: 1509


In [68]:
duplicate_timestamp_count = sample_minute_df.duplicated(subset=["timestamp"]).sum()

print("Duplicate timestamps:", duplicate_timestamp_count)

Duplicate timestamps: 0


In [69]:
sample_minute_missingness = sample_minute_df.isna().sum().to_frame("missing_count")

sample_minute_missingness["missing_pct"] = (sample_minute_missingness["missing_count"] / len(sample_minute_df) * 100)

sample_minute_missingness

,missing_count,missing_pct
timestamp,0,0.0000
open,0,0.0000
high,0,0.0000
low,0,0.0000
close,0,0.0000
volume,0,0.0000
date,0,0.0000
time,0,0.0000
symbol,0,0.0000


In [70]:
print("Earliest observed bar time:", sample_minute_df["time"].min())
print("Latest observed bar time:", sample_minute_df["time"].max())

Earliest observed bar time: 09:15:00
Latest observed bar time: 19:14:00


In [71]:
bars_per_day = sample_minute_df.groupby("date").size()

bars_per_day.describe()

count   1,509.0000
mean      373.3976
std        21.4128
min        60.0000
25%       375.0000
50%       375.0000
75%       375.0000
max       375.0000
dtype: float64

In [72]:
bars_per_day.value_counts().sort_index()

60        5
105       2
149       1
330       1
360       1
370       1
372       2
373       1
374       4
375    1491
Name: count, dtype: int64

In [73]:
incomplete_session_dates = bars_per_day[bars_per_day != 375]

print("Incomplete sessions:", len(incomplete_session_dates))

incomplete_session_dates.head(10)

Incomplete sessions: 18


date
2020-08-27    373
2020-11-02    374
2020-11-14     60
2021-02-24    149
2021-11-04     60
2021-12-20    374
2022-03-07    360
2022-10-24     60
2023-07-12    374
2023-07-20    330
dtype: int64

In [74]:
complete_dates = bars_per_day[bars_per_day == 375].index

sample_complete_date = complete_dates[0]

sample_complete_session = sample_minute_df.loc[sample_minute_df["date"] == sample_complete_date].copy()

print("Sample complete date:", sample_complete_date)
print("Bars:", len(sample_complete_session))
print("First timestamp:", sample_complete_session["timestamp"].min())
print("Last timestamp:", sample_complete_session["timestamp"].max())

sample_complete_session.head()

Sample complete date: 2020-06-01 00:00:00
Bars: 375
First timestamp: 2020-06-01 09:15:00
Last timestamp: 2020-06-01 15:29:00


,timestamp,open,high,low,close,volume,date,time,symbol
0,2020-06-01 09:15:00,705.4000,706.0500,703.4500,704.7000,772062,2020-06-01,09:15:00,RELIANCE
1,2020-06-01 09:16:00,704.7000,705.6000,703.8000,703.9000,361144,2020-06-01,09:16:00,RELIANCE
2,2020-06-01 09:17:00,703.8000,706.3000,703.7000,706.3000,310700,2020-06-01,09:17:00,RELIANCE
3,2020-06-01 09:18:00,706.8000,708.6500,706.0000,708.1000,265712,2020-06-01,09:18:00,RELIANCE
4,2020-06-01 09:19:00,708.5000,709.9000,707.0000,708.8000,256512,2020-06-01,09:19:00,RELIANCE


In [75]:
sample_complete_session["minute_return"] = sample_complete_session["close"].pct_change()

In [76]:
intraday_realized_volatility = np.sqrt(np.square(sample_complete_session["minute_return"].dropna()).sum())

intraday_realized_volatility

np.float64(0.01767422356450869)

In [77]:
session_open = sample_complete_session["open"].iloc[0]
session_close = sample_complete_session["close"].iloc[-1]

full_session_return = session_close / session_open - 1

print("Session open:", session_open)
print("Session close:", session_close)
print("Full-session return:", full_session_return)

Session open: 705.4
Session close: 724.9
Full-session return: 0.02764388999149414


In [78]:
last_30_minute_rows = sample_complete_session.tail(30).copy()

print("First bar in final 30 minutes:", last_30_minute_rows["timestamp"].min())
print("Last bar in final 30 minutes:", last_30_minute_rows["timestamp"].max())
print("Rows:", len(last_30_minute_rows))

First bar in final 30 minutes: 2020-06-01 15:00:00
Last bar in final 30 minutes: 2020-06-01 15:29:00
Rows: 30


In [79]:
last_30_start_price = last_30_minute_rows["open"].iloc[0]
last_30_end_price = last_30_minute_rows["close"].iloc[-1]

last_30_minute_return = last_30_end_price / last_30_start_price - 1

print("Final 30-minute return:", last_30_minute_return)

Final 30-minute return: 0.002212083506152318


In [80]:
if abs(full_session_return) > 1e-12:
    close_auction_return_concentration = last_30_minute_return / full_session_return
else:
    close_auction_return_concentration = np.nan

close_auction_return_concentration

np.float64(0.0800207028328127)

In [81]:
full_session_volume = sample_complete_session["volume"].sum()
last_30_minute_volume = last_30_minute_rows["volume"].sum()

if full_session_volume > 0:
    close_auction_volume_concentration = last_30_minute_volume / full_session_volume
else:
    close_auction_volume_concentration = np.nan

close_auction_volume_concentration

np.float64(0.10365402138104447)

In [82]:
session_midpoint = len(sample_complete_session) // 2

morning_session = sample_complete_session.iloc[:session_midpoint].copy()
afternoon_session = sample_complete_session.iloc[session_midpoint:].copy()

print("Morning bars:", len(morning_session))
print("Afternoon bars:", len(afternoon_session))

Morning bars: 187
Afternoon bars: 188


In [83]:
morning_return = (morning_session["close"].iloc[-1] / morning_session["open"].iloc[0] - 1)

afternoon_return = (afternoon_session["close"].iloc[-1] / afternoon_session["open"].iloc[0] - 1)

morning_vs_afternoon_return = morning_return - afternoon_return

print("Morning return:", morning_return)
print("Afternoon return:", afternoon_return)
print("Morning minus afternoon:", morning_vs_afternoon_return)

Morning return: 0.03473206691239006
Afternoon return: -0.006850253459377953
Morning minus afternoon: 0.04158232037176801


In [84]:
session_typical_price = (sample_complete_session["high"]+ sample_complete_session["low"]+ sample_complete_session["close"]) / 3

session_vwap_numerator = (session_typical_price * sample_complete_session["volume"]).sum()

session_vwap_denominator = sample_complete_session["volume"].sum()

if session_vwap_denominator > 0:
    session_vwap = session_vwap_numerator / session_vwap_denominator
else:
    session_vwap = np.nan

session_vwap

np.float64(721.1028974456108)

In [85]:
if pd.notna(session_vwap) and session_vwap != 0:
    close_vwap_deviation = session_close / session_vwap - 1
else:
    close_vwap_deviation = np.nan

close_vwap_deviation

np.float64(0.005265687556990528)

In [86]:
valid_amihud_rows = sample_complete_session[
    sample_complete_session["minute_return"].notna()
    & (sample_complete_session["volume"] > 0)
].copy()

valid_amihud_rows["amihud_component"] = (
    valid_amihud_rows["minute_return"].abs()
    / valid_amihud_rows["volume"]
)

amihud_illiquidity = valid_amihud_rows["amihud_component"].mean()

amihud_illiquidity

np.float64(7.90294214177709e-09)

In [87]:
sample_minute_features = pd.Series(
    {
        "intraday_realized_volatility_pct": intraday_realized_volatility * 100,
        "close_auction_return_concentration": close_auction_return_concentration,
        "close_auction_volume_concentration": close_auction_volume_concentration,
        "morning_vs_afternoon_return_pct": morning_vs_afternoon_return * 100,
        "close_vwap_deviation_pct": close_vwap_deviation * 100,
        "amihud_illiquidity": amihud_illiquidity,
        "minute_bar_count": len(sample_complete_session),
        "is_complete_session": len(sample_complete_session) == 375,
    }
)

sample_minute_features

intraday_realized_volatility_pct     1.7674
close_auction_return_concentration   0.0800
close_auction_volume_concentration   0.1037
morning_vs_afternoon_return_pct      4.1582
close_vwap_deviation_pct             0.5266
amihud_illiquidity                   0.0000
minute_bar_count                        375
is_complete_session                    True
dtype: object

### Build Reusable Minute-to-Daily Feature Function

The following function converts one stock's raw minute bars into one row per trading date.

In [88]:
def create_minute_daily_features(df: pd.DataFrame, symbol: str) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(column).strip().lower() for column in df.columns]
    required_columns = {"timestamp", "open", "high", "low", "close", "volume"}
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{symbol} minute data is missing columns: {sorted(missing_columns)}"
        )

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="raise")

    df = df.sort_values("timestamp").reset_index(drop=True)

    df["pred_date"] = df["timestamp"].dt.normalize()

    daily_rows = []

    for pred_date, session_df in df.groupby("pred_date", sort=True):
        session_df = session_df.sort_values("timestamp").reset_index(drop=True)

        minute_bar_count = len(session_df)
        is_complete_session = minute_bar_count == 375

        session_df["minute_return"] = session_df["close"].pct_change()

        valid_minute_returns = session_df["minute_return"].dropna()

        if len(valid_minute_returns) > 0:
            intraday_realized_volatility_pct = (
                np.sqrt(np.square(valid_minute_returns).sum()) * 100
            )
        else:
            intraday_realized_volatility_pct = np.nan

        session_open = session_df["open"].iloc[0]
        session_close = session_df["close"].iloc[-1]

        if session_open > 0:
            full_session_return = session_close / session_open - 1
        else:
            full_session_return = np.nan

        last_30_rows = session_df.tail(min(30, minute_bar_count)).copy()

        if len(last_30_rows) > 0 and last_30_rows["open"].iloc[0] > 0:
            last_30_return = (
                last_30_rows["close"].iloc[-1]
                / last_30_rows["open"].iloc[0]
                - 1
            )
        else:
            last_30_return = np.nan

        if pd.notna(full_session_return) and abs(full_session_return) > 1e-12:
            close_auction_return_concentration = (
                last_30_return / full_session_return
            )
        else:
            close_auction_return_concentration = np.nan

        full_session_volume = session_df["volume"].sum()
        last_30_volume = last_30_rows["volume"].sum()

        if full_session_volume > 0:
            close_auction_volume_concentration = (
                last_30_volume / full_session_volume
            )
        else:
            close_auction_volume_concentration = np.nan

        session_midpoint = minute_bar_count // 2

        morning_session = session_df.iloc[:session_midpoint]
        afternoon_session = session_df.iloc[session_midpoint:]

        if (
            len(morning_session) > 0
            and morning_session["open"].iloc[0] > 0
        ):
            morning_return = (
                morning_session["close"].iloc[-1]
                / morning_session["open"].iloc[0]
                - 1
            )
        else:
            morning_return = np.nan

        if (
            len(afternoon_session) > 0
            and afternoon_session["open"].iloc[0] > 0
        ):
            afternoon_return = (
                afternoon_session["close"].iloc[-1]
                / afternoon_session["open"].iloc[0]
                - 1
            )
        else:
            afternoon_return = np.nan

        if pd.notna(morning_return) and pd.notna(afternoon_return):
            morning_vs_afternoon_return_pct = (
                morning_return - afternoon_return
            ) * 100
        else:
            morning_vs_afternoon_return_pct = np.nan

        typical_price = (
            session_df["high"]
            + session_df["low"]
            + session_df["close"]
        ) / 3

        vwap_denominator = session_df["volume"].sum()

        if vwap_denominator > 0:
            session_vwap = (
                typical_price * session_df["volume"]
            ).sum() / vwap_denominator
        else:
            session_vwap = np.nan

        if pd.notna(session_vwap) and session_vwap != 0:
            close_vwap_deviation_pct = (
                session_close / session_vwap - 1
            ) * 100
        else:
            close_vwap_deviation_pct = np.nan

        valid_amihud_rows = session_df[
            session_df["minute_return"].notna()
            & (session_df["volume"] > 0)
        ]

        if len(valid_amihud_rows) > 0:
            amihud_illiquidity = (
                valid_amihud_rows["minute_return"].abs()
                / valid_amihud_rows["volume"]
            ).mean()
        else:
            amihud_illiquidity = np.nan

        daily_rows.append(
            {
                "pred_date": pred_date,
                "symbol": symbol,
                "intraday_realized_volatility_pct": intraday_realized_volatility_pct,
                "close_auction_return_concentration": close_auction_return_concentration,
                "close_auction_volume_concentration": close_auction_volume_concentration,
                "morning_vs_afternoon_return_pct": morning_vs_afternoon_return_pct,
                "close_vwap_deviation_pct": close_vwap_deviation_pct,
                "amihud_illiquidity": amihud_illiquidity,
                "minute_bar_count": minute_bar_count,
                "is_complete_session": is_complete_session,
            }
        )

    return pd.DataFrame(daily_rows)

In [89]:
reliance_minute_features = create_minute_daily_features(
    sample_minute_df,
    sample_minute_symbol,
)

print("Shape:", reliance_minute_features.shape)

reliance_minute_features.head()

Shape: (1509, 10)


,pred_date,symbol,intraday_realized_volatility_pct,close_auction_return_concentration,close_auction_volume_concentration,morning_vs_afternoon_return_pct,close_vwap_deviation_pct,amihud_illiquidity,minute_bar_count,is_complete_session
0,2020-06-01,RELIANCE,1.7674,0.0800,0.1037,4.1582,0.5266,0.0000,375,True
1,2020-06-02,RELIANCE,1.2389,-0.0361,0.1175,-0.4804,0.3817,0.0000,375,True
2,2020-06-03,RELIANCE,1.3994,-0.4218,0.1377,-0.0747,-0.0510,0.0000,375,True
3,2020-06-04,RELIANCE,1.7369,0.0678,0.1323,1.9174,0.3635,0.0000,375,True
4,2020-06-05,RELIANCE,1.7536,0.4926,0.1436,1.0063,-1.0772,0.0000,375,True


In [90]:
reliance_minute_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1509 entries, 0 to 1508
Data columns (total 10 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   pred_date                           1509 non-null   datetime64[ns]
 1   symbol                              1509 non-null   object        
 2   intraday_realized_volatility_pct    1509 non-null   float64       
 3   close_auction_return_concentration  1500 non-null   float64       
 4   close_auction_volume_concentration  1509 non-null   float64       
 5   morning_vs_afternoon_return_pct     1509 non-null   float64       
 6   close_vwap_deviation_pct            1509 non-null   float64       
 7   amihud_illiquidity                  1509 non-null   float64       
 8   minute_bar_count                    1509 non-null   int64         
 9   is_complete_session                 1509 non-null   bool          
dtypes: bool(1), datetime64[n

In [91]:
duplicate_count = reliance_minute_features.duplicated(subset=["symbol", "pred_date"]).sum()

print("Duplicate symbol-date rows:", duplicate_count)

Duplicate symbol-date rows: 0


In [92]:
reliance_minute_missingness = (reliance_minute_features.isna().sum().to_frame("missing_count"))
reliance_minute_missingness["missing_pct"] = (reliance_minute_missingness["missing_count"]/ len(reliance_minute_features)* 100)
reliance_minute_missingness

,missing_count,missing_pct
pred_date,0,0.0000
symbol,0,0.0000
intraday_realized_volatility_pct,0,0.0000
close_auction_return_concentration,9,0.5964
close_auction_volume_concentration,0,0.0000
morning_vs_afternoon_return_pct,0,0.0000
close_vwap_deviation_pct,0,0.0000
amihud_illiquidity,0,0.0000
minute_bar_count,0,0.0000
is_complete_session,0,0.0000


In [93]:
reliance_minute_features.loc[~reliance_minute_features["is_complete_session"]].head(10)

,pred_date,symbol,intraday_realized_volatility_pct,close_auction_return_concentration,close_auction_volume_concentration,morning_vs_afternoon_return_pct,close_vwap_deviation_pct,amihud_illiquidity,minute_bar_count,is_complete_session
63,2020-08-27,RELIANCE,1.4034,0.1177,0.1658,0.7401,-0.8905,0.0000,373,False
109,2020-11-02,RELIANCE,2.6319,0.1513,0.1546,-3.2539,-2.6313,0.0000,374,False
119,2020-11-14,RELIANCE,0.5300,0.3702,0.3918,-0.1401,-0.2250,0.0000,60,False
188,2021-02-24,RELIANCE,0.9209,-0.3192,0.0854,0.4603,-0.2069,0.0000,149,False
359,2021-11-04,RELIANCE,0.3145,0.5078,0.3607,-0.0079,0.2907,0.0000,60,False
389,2021-12-20,RELIANCE,1.5070,-0.1251,0.1694,-1.7644,-0.1098,0.0000,374,False
442,2022-03-07,RELIANCE,2.0909,-0.0312,0.1334,0.6458,-0.6973,0.0000,360,False
599,2022-10-24,RELIANCE,0.8324,0.0203,0.2475,0.3738,-0.1259,0.0000,60,False
776,2023-07-12,RELIANCE,1.0654,6.0651,0.2460,1.4285,-0.6373,0.0000,374,False
782,2023-07-20,RELIANCE,1.1481,0.2144,0.1058,0.3060,0.2863,0.0000,330,False


In [94]:
minute_feature_frames = []
minute_processing_errors = []

for index, file_path in enumerate(minute_files, start=1):
    symbol = file_path.stem

    try:
        stock_minute_df = pd.read_parquet(file_path)

        stock_minute_features = create_minute_daily_features(
            stock_minute_df,
            symbol,
        )

        minute_feature_frames.append(stock_minute_features)

    except Exception as error:
        minute_processing_errors.append(
            {
                "symbol": symbol,
                "file": file_path.name,
                "error": str(error),
            }
        )

    if index % 20 == 0 or index == len(minute_files):
        print(f"Processed {index}/{len(minute_files)} files")

Processed 20/208 files
Processed 40/208 files
Processed 60/208 files
Processed 80/208 files
Processed 100/208 files
Processed 120/208 files
Processed 140/208 files
Processed 160/208 files
Processed 180/208 files
Processed 200/208 files
Processed 208/208 files


In [95]:
print("Files processed successfully:", len(minute_feature_frames))
print("Files with errors:", len(minute_processing_errors))

Files processed successfully: 208
Files with errors: 0


In [96]:
minute_processing_errors_df = pd.DataFrame(minute_processing_errors)

minute_processing_errors_df

""


In [97]:
minute_features_df = pd.concat(
    minute_feature_frames,
    ignore_index=True,
)

minute_features_df = minute_features_df.sort_values(
    ["symbol", "pred_date"]
).reset_index(drop=True)

print("Combined minute feature shape:", minute_features_df.shape)

minute_features_df.head()

Combined minute feature shape: (301074, 10)


,pred_date,symbol,intraday_realized_volatility_pct,close_auction_return_concentration,close_auction_volume_concentration,morning_vs_afternoon_return_pct,close_vwap_deviation_pct,amihud_illiquidity,minute_bar_count,is_complete_session
0,2020-06-01,360ONE,6.7210,-0.7783,0.1444,0.0941,-3.0433,0.0000,370,False
1,2020-06-02,360ONE,3.9752,14.0406,0.4056,0.0222,0.1430,0.0001,258,False
2,2020-06-03,360ONE,4.4413,NaN,0.2501,-1.0959,0.3693,0.0001,254,False
3,2020-06-04,360ONE,7.6043,-0.9689,0.1259,0.3683,-0.6457,0.0001,301,False
4,2020-06-05,360ONE,4.9324,0.0172,0.1136,-5.7220,1.7953,0.0001,317,False


In [98]:
print("Symbols:", minute_features_df["symbol"].nunique())
print("Trading dates:", minute_features_df["pred_date"].nunique())

print("Date range:",minute_features_df["pred_date"].min(),"to",minute_features_df["pred_date"].max(),)

Symbols: 208
Trading dates: 1509
Date range: 2020-06-01 00:00:00 to 2026-06-29 00:00:00


In [99]:
minute_duplicate_count = minute_features_df.duplicated(
    subset=["symbol", "pred_date"]
).sum()

print("Duplicate symbol-date rows:", minute_duplicate_count)

Duplicate symbol-date rows: 0


In [100]:
minute_feature_columns = [
    "intraday_realized_volatility_pct",
    "close_auction_return_concentration",
    "close_auction_volume_concentration",
    "morning_vs_afternoon_return_pct",
    "close_vwap_deviation_pct",
    "amihud_illiquidity",
]


minute_quality_columns = [
    "minute_bar_count",
    "is_complete_session",
]

In [101]:
minute_feature_missingness = minute_features_df[
    minute_feature_columns + minute_quality_columns
].isna().sum().to_frame("missing_count")

minute_feature_missingness["missing_pct"] = (
    minute_feature_missingness["missing_count"]
    / len(minute_features_df)
    * 100
)

minute_feature_missingness.sort_values(
    "missing_pct",
    ascending=False,
)

,missing_count,missing_pct
close_auction_return_concentration,3591,1.1927
intraday_realized_volatility_pct,0,0.0000
close_auction_volume_concentration,0,0.0000
morning_vs_afternoon_return_pct,0,0.0000
close_vwap_deviation_pct,0,0.0000
amihud_illiquidity,0,0.0000
minute_bar_count,0,0.0000
is_complete_session,0,0.0000


In [102]:
minute_features_df[minute_feature_columns] = (minute_features_df[minute_feature_columns].replace([np.inf, -np.inf], np.nan))

In [103]:
minute_features_df["is_complete_session"].value_counts(dropna=False)

is_complete_session
True     289642
False     11432
Name: count, dtype: int64

In [104]:
minute_features_df["minute_bar_count"].describe()

count   301,074.0000
mean        372.9692
std          22.1926
min          45.0000
25%         375.0000
50%         375.0000
75%         375.0000
max         375.0000
Name: minute_bar_count, dtype: float64

In [105]:
minute_features_df["minute_bar_count"].value_counts().sort_index().tail(10)

minute_bar_count
366       146
367       173
368       235
369       353
370       463
371       473
372       887
373      1243
374      3009
375    289642
Name: count, dtype: int64

In [106]:
assert minute_duplicate_count == 0

assert (minute_features_df["minute_bar_count"] > 0).all()

assert minute_features_df["is_complete_session"].isin([True, False]).all()

valid_volume_concentration = minute_features_df["close_auction_volume_concentration"].dropna()

assert valid_volume_concentration.between(0, 1).all()

print("Minute feature sanity checks passed.")

Minute feature sanity checks passed.


In [107]:
minute_features_path = (PROCESSED_DATA_DIR / "minute_features_v1.parquet")

minute_features_df.to_parquet(minute_features_path,index=False,)

print("Saved minute feature panel to:")
print(minute_features_path)

Saved minute feature panel to:
/Users/kushagr/Desktop/astra-assignment/outputs/processed_data/minute_features_v1.parquet


### Merge Daily and Minute Feature Panels

In [108]:
print(daily_features_df[["symbol", "pred_date"]].head())

print(minute_features_df[["symbol", "pred_date"]].head())

   symbol  pred_date
0  360ONE 2020-06-01
1  360ONE 2020-06-02
2  360ONE 2020-06-03
3  360ONE 2020-06-04
4  360ONE 2020-06-05
   symbol  pred_date
0  360ONE 2020-06-01
1  360ONE 2020-06-02
2  360ONE 2020-06-03
3  360ONE 2020-06-04
4  360ONE 2020-06-05


In [109]:
print(daily_features_df[["symbol", "pred_date"]].dtypes)

print(minute_features_df[["symbol", "pred_date"]].dtypes)

symbol               object
pred_date    datetime64[ns]
dtype: object
symbol               object
pred_date    datetime64[ns]
dtype: object


In [110]:
daily_keys = daily_features_df[["symbol", "pred_date"]].drop_duplicates()

minute_keys = minute_features_df[["symbol", "pred_date"]].drop_duplicates()

matching_keys = daily_keys.merge(minute_keys,on=["symbol", "pred_date"],how="inner",)

print("Daily stock-date keys:", len(daily_keys))
print("Minute stock-date keys:", len(minute_keys))
print("Matching stock-date keys:", len(matching_keys))
print("Daily coverage matched by minute data: {:.2f}%".format(len(matching_keys) / len(daily_keys) * 100))

Daily stock-date keys: 301275
Minute stock-date keys: 301074
Matching stock-date keys: 301074
Daily coverage matched by minute data: 99.93%


In [111]:
model_df = daily_features_df.merge(minute_features_df,on=["symbol", "pred_date"],how="left",validate="one_to_one",)

print("Merged model dataset shape:", model_df.shape)

model_df.head()

Merged model dataset shape: (301275, 37)


,pred_date,symbol,open,high,low,close,volume,target_date,next_open,actual_return_pct,actual_direction,actual_magnitude_pct,calendar_gap_days,lagged_overnight_return_1d,gap_mean_20d,overnight_std_20d,gap_positive_fraction_20d,return_1d,return_5d,return_20d,daily_volatility_5d,daily_volatility_20d,volume_mean_20d_lagged,volume_std_20d_lagged,volume_zscore_20d,volume_mean_5d,volume_mean_20d,volume_trend_5d_20d,day_of_week,intraday_realized_volatility_pct,close_auction_return_concentration,close_auction_volume_concentration,morning_vs_afternoon_return_pct,close_vwap_deviation_pct,amihud_illiquidity,minute_bar_count,is_complete_session
0,2020-06-01,360ONE,209.1500,230.9500,209.1500,219.2500,185884,2020-06-02,224.9500,2.5998,1,2.5998,1.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,6.7210,-0.7783,0.1444,0.0941,-3.0433,0.0000,370.0000,False
1,2020-06-02,360ONE,224.9500,228.7500,218.9000,224.6500,22368,2020-06-03,228.7500,1.8251,1,1.8251,1.0000,2.5998,NaN,NaN,NaN,2.4629,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,3.9752,14.0406,0.4056,0.0222,0.1430,0.0001,258.0000,False
2,2020-06-03,360ONE,228.7500,230.0000,221.2500,227.7000,34560,2020-06-04,233.7500,2.6570,1,2.6570,1.0000,1.8251,NaN,NaN,NaN,1.3577,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,4.4413,NaN,0.2501,-1.0959,0.3693,0.0001,254.0000,False
3,2020-06-04,360ONE,233.7500,247.5000,228.7500,239.8500,77532,2020-06-05,244.0500,1.7511,1,1.7511,1.0000,2.6570,NaN,NaN,NaN,5.3360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,7.6043,-0.9689,0.1259,0.3683,-0.6457,0.0001,301.0000,False
4,2020-06-05,360ONE,244.0500,253.0500,236.5000,250.1500,22376,2020-06-08,251.7500,0.6396,1,0.6396,3.0000,1.7511,NaN,NaN,NaN,4.2944,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"68,544.0000",NaN,NaN,4,4.9324,0.0172,0.1136,-5.7220,1.7953,0.0001,317.0000,False


In [112]:
print("Daily rows before merge:", len(daily_features_df))
print("Rows after merge:", len(model_df))

assert len(model_df) == len(daily_features_df)

print("All daily rows were preserved.")

Daily rows before merge: 301275
Rows after merge: 301275
All daily rows were preserved.


In [113]:
minute_merge_coverage = model_df[minute_feature_columns].notna().any(axis=1).mean() * 100

print("Rows with at least one minute feature: {:.2f}%".format(minute_merge_coverage))

Rows with at least one minute feature: 99.93%


In [114]:
unmatched_minute_rows = model_df[model_df[minute_feature_columns].isna().all(axis=1)]

print("Rows without minute features:", len(unmatched_minute_rows))

unmatched_minute_rows[["pred_date", "symbol", "open", "close"]].head(10)

Rows without minute features: 201


,pred_date,symbol,open,close
860,2023-11-12,360ONE,545.0000,531.9500
2370,2023-11-12,ABB,"4,265.9500","4,235.1500"
3880,2023-11-12,ABCAPITAL,176.3500,175.6000
5390,2023-11-12,ADANIENSOL,772.4500,768.4000
6900,2023-11-12,ADANIENT,"2,152.0000","2,143.9000"
8410,2023-11-12,ADANIGREEN,942.9500,948.1000
9920,2023-11-12,ADANIPORTS,816.7000,812.2500
11430,2023-11-12,ADANIPOWER,80.7000,79.9500
12940,2023-11-12,ALKEM,"4,330.0000","4,316.2000"
14450,2023-11-12,AMBER,"3,135.0000","3,160.1500"


In [115]:
unmatched_by_symbol = (unmatched_minute_rows.groupby("symbol").size().sort_values(ascending=False))

unmatched_by_symbol.head(20)

symbol
360ONE        1
NYKAA         1
MPHASIS       1
MUTHOOTFIN    1
NAM-INDIA     1
NATIONALUM    1
NAUKRI        1
NBCC          1
NESTLEIND     1
NHPC          1
NMDC          1
NTPC          1
OBEROIRLTY    1
JSWSTEEL      1
OFSS          1
OIL           1
ONGC          1
PAGEIND       1
PATANJALI     1
PAYTM         1
dtype: int64

In [116]:
model_duplicate_count = model_df.duplicated(subset=["symbol", "pred_date"]).sum()

print("Duplicate symbol-date rows after merge:", model_duplicate_count)

assert model_duplicate_count == 0

Duplicate symbol-date rows after merge: 0


In [117]:
all_feature_columns = daily_feature_columns + minute_feature_columns

print("Daily features:", len(daily_feature_columns))
print("Minute features:", len(minute_feature_columns))
print("Total Version 1 features so far:", len(all_feature_columns))

Daily features: 13
Minute features: 6
Total Version 1 features so far: 19


In [118]:
combined_feature_missingness = model_df[all_feature_columns].isna().sum().to_frame("missing_count")

combined_feature_missingness["missing_pct"] = (combined_feature_missingness["missing_count"] / len(model_df) * 100)

combined_feature_missingness.sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
volume_zscore_20d,4160,1.3808
return_20d,4160,1.3808
daily_volatility_20d,4160,1.3808
overnight_std_20d,4160,1.3808
gap_mean_20d,4160,1.3808
gap_positive_fraction_20d,4160,1.3808
volume_trend_5d_20d,3952,1.3118
close_auction_return_concentration,3792,1.2587
return_5d,1040,0.3452
daily_volatility_5d,1040,0.3452


In [119]:
merged_panel_path = PROCESSED_DATA_DIR / "model_panel_daily_minute_v1.parquet"

model_df.to_parquet(merged_panel_path,index=False,)

print("Saved merged daily-minute panel to:")
print(merged_panel_path)

Saved merged daily-minute panel to:
/Users/kushagr/Desktop/astra-assignment/outputs/processed_data/model_panel_daily_minute_v1.parquet


### Cross-Sectional Feature Engineering — Version 1

In [120]:
cross_sectional_df = model_df.copy()

cross_sectional_df = cross_sectional_df.sort_values(["pred_date", "symbol"]).reset_index(drop=True)

print("Starting shape:", cross_sectional_df.shape)

Starting shape: (301275, 37)


In [121]:
cross_sectional_df["return_1d_rank_pct"] = (cross_sectional_df.groupby("pred_date")["return_1d"].rank(method="average", pct=True))

In [122]:
return_mean_by_date = (cross_sectional_df.groupby("pred_date")["return_1d"].transform("mean"))

return_std_by_date = (cross_sectional_df.groupby("pred_date")["return_1d"].transform("std"))

cross_sectional_df["return_1d_zscore"] = (cross_sectional_df["return_1d"] - return_mean_by_date) / return_std_by_date

In [123]:
cross_sectional_df["universe_mean_return_1d"] = (cross_sectional_df.groupby("pred_date")["return_1d"].transform("mean"))

cross_sectional_df["cross_sectional_return_dispersion"] = (cross_sectional_df.groupby("pred_date")["return_1d"].transform("std"))

cross_sectional_df["market_breadth"] = (cross_sectional_df["return_1d"].gt(0).groupby(cross_sectional_df["pred_date"]).transform("mean"))

cross_sectional_df["aggregate_universe_volatility"] = (cross_sectional_df.groupby("pred_date")["daily_volatility_20d"].transform("median"))

In [124]:
universe_return_df = (cross_sectional_df[["pred_date", "universe_mean_return_1d"]].drop_duplicates(subset=["pred_date"]).sort_values("pred_date").reset_index(drop=True))
universe_return_df.head()

,pred_date,universe_mean_return_1d
0,2020-06-01,NaN
1,2020-06-02,1.7718
2,2020-06-03,1.0758
3,2020-06-04,0.0977
4,2020-06-05,2.6821


In [125]:
cross_sectional_df["stock_market_product"] = (cross_sectional_df["return_1d"] * cross_sectional_df["universe_mean_return_1d"])

cross_sectional_df["market_return_squared"] = (cross_sectional_df["universe_mean_return_1d"] ** 2)

In [126]:
cross_sectional_df["rolling_mean_stock_return_60d"] = (cross_sectional_df.groupby("symbol")["return_1d"].transform(
        lambda series: series.rolling(
            window=60,
            min_periods=40,
        ).mean()
    )
)

cross_sectional_df["rolling_mean_market_return_60d"] = (
    cross_sectional_df.groupby("symbol")["universe_mean_return_1d"]
    .transform(
        lambda series: series.rolling(
            window=60,
            min_periods=40,
        ).mean()
    )
)

cross_sectional_df["rolling_mean_stock_market_product_60d"] = (
    cross_sectional_df.groupby("symbol")["stock_market_product"]
    .transform(
        lambda series: series.rolling(
            window=60,
            min_periods=40,
        ).mean()
    )
)

In [127]:
cross_sectional_df["rolling_mean_market_squared_60d"] = (
    cross_sectional_df.groupby("symbol")["market_return_squared"]
    .transform(
        lambda series: series.rolling(
            window=60,
            min_periods=40,
        ).mean()
    )
)

cross_sectional_df["rolling_market_variance_60d"] = (
    cross_sectional_df["rolling_mean_market_squared_60d"]
    - cross_sectional_df["rolling_mean_market_return_60d"] ** 2
)

In [128]:
cross_sectional_df["rolling_stock_market_covariance_60d"] = (
    cross_sectional_df["rolling_mean_stock_market_product_60d"]
    - (
        cross_sectional_df["rolling_mean_stock_return_60d"]
        * cross_sectional_df["rolling_mean_market_return_60d"]
    )
)

cross_sectional_df["rolling_beta_to_universe_60d"] = (
    cross_sectional_df["rolling_stock_market_covariance_60d"]
    / cross_sectional_df["rolling_market_variance_60d"]
)

In [129]:
cross_sectional_feature_columns = ["return_1d_rank_pct","return_1d_zscore","rolling_beta_to_universe_60d","cross_sectional_return_dispersion","market_breadth","aggregate_universe_volatility",]
print("Cross-sectional features:", len(cross_sectional_feature_columns))

Cross-sectional features: 6


In [130]:
cross_sectional_df[cross_sectional_feature_columns] = (cross_sectional_df[cross_sectional_feature_columns].replace([np.inf, -np.inf], np.nan))

In [131]:
cross_sectional_missingness = (
    cross_sectional_df[cross_sectional_feature_columns]
    .isna()
    .sum()
    .to_frame("missing_count")
)

cross_sectional_missingness["missing_pct"] = (
    cross_sectional_missingness["missing_count"]
    / len(cross_sectional_df)
    * 100
)

cross_sectional_missingness.sort_values(
    "missing_pct",
    ascending=False,
)

,missing_count,missing_pct
rolling_beta_to_universe_60d,8320,2.7616
aggregate_universe_volatility,3680,1.2215
return_1d_rank_pct,208,0.0690
return_1d_zscore,208,0.0690
cross_sectional_return_dispersion,184,0.0611
market_breadth,0,0.0000


In [132]:
valid_return_ranks = cross_sectional_df["return_1d_rank_pct"].dropna()
valid_market_breadth = cross_sectional_df["market_breadth"].dropna()
valid_dispersion = cross_sectional_df["cross_sectional_return_dispersion"].dropna()
valid_aggregate_volatility = cross_sectional_df["aggregate_universe_volatility"].dropna()

assert valid_return_ranks.between(0, 1).all()
assert valid_market_breadth.between(0, 1).all()
assert (valid_dispersion >= 0).all()
assert (valid_aggregate_volatility >= 0).all()

print("Cross-sectional feature sanity checks passed.")

Cross-sectional feature sanity checks passed.


In [133]:
sample_cross_sectional_date = (
    cross_sectional_df["pred_date"]
    .dropna()
    .sort_values()
    .iloc[-100]
)

cross_sectional_df.loc[
    cross_sectional_df["pred_date"] == sample_cross_sectional_date,
    [
        "pred_date",
        "symbol",
        "return_1d",
        "return_1d_rank_pct",
        "return_1d_zscore",
        "universe_mean_return_1d",
        "cross_sectional_return_dispersion",
        "market_breadth",
        "aggregate_universe_volatility",
        "rolling_beta_to_universe_60d",
    ],
].sort_values("return_1d", ascending=False).head(10)

,pred_date,symbol,return_1d,return_1d_rank_pct,return_1d_zscore,universe_mean_return_1d,cross_sectional_return_dispersion,market_breadth,aggregate_universe_volatility,rolling_beta_to_universe_60d
301201,2026-06-29,NATIONALUM,4.5913,1.0000,2.6045,-0.5981,1.9925,0.3846,1.7865,0.4638
301268,2026-06-29,VEDL,4.0592,0.9952,2.3374,-0.5981,1.9925,0.3846,1.7865,1.4011
301175,2026-06-29,KEI,3.4928,0.9904,2.0531,-0.5981,1.9925,0.3846,1.7865,1.2782
301179,2026-06-29,LAURUSLABS,3.4690,0.9856,2.0412,-0.5981,1.9925,0.3846,1.7865,0.4800
301259,2026-06-29,TORNTPHARM,3.1310,0.9808,1.8715,-0.5981,1.9925,0.3846,1.7865,0.1107
301228,2026-06-29,POWERINDIA,2.9736,0.9760,1.7926,-0.5981,1.9925,0.3846,1.7865,0.7375
301194,2026-06-29,MCX,2.9240,0.9712,1.7677,-0.5981,1.9925,0.3846,1.7865,0.6821
301186,2026-06-29,LUPIN,2.8078,0.9663,1.7093,-0.5981,1.9925,0.3846,1.7865,0.3022
301098,2026-06-29,BHEL,2.7440,0.9615,1.6773,-0.5981,1.9925,0.3846,1.7865,1.1894
301205,2026-06-29,NHPC,2.7330,0.9567,1.6718,-0.5981,1.9925,0.3846,1.7865,0.8698


In [134]:
all_feature_columns = (
    daily_feature_columns
    + minute_feature_columns
    + cross_sectional_feature_columns
)

print("Daily features:", len(daily_feature_columns))
print("Minute features:", len(minute_feature_columns))
print("Cross-sectional features:", len(cross_sectional_feature_columns))
print("Total Version 1 features:", len(all_feature_columns))

Daily features: 13
Minute features: 6
Cross-sectional features: 6
Total Version 1 features: 25


In [135]:
temporary_cross_sectional_columns = [
    "stock_market_product",
    "market_return_squared",
    "rolling_mean_stock_return_60d",
    "rolling_mean_market_return_60d",
    "rolling_mean_stock_market_product_60d",
    "rolling_mean_market_squared_60d",
    "rolling_market_variance_60d",
    "rolling_stock_market_covariance_60d",
]

cross_sectional_df = cross_sectional_df.drop(
    columns=temporary_cross_sectional_columns
)

In [136]:
model_features_df = cross_sectional_df.copy()

print("Completed feature panel shape:", model_features_df.shape)

Completed feature panel shape: (301275, 44)


In [137]:
complete_feature_rows = model_features_df.dropna(
    subset=all_feature_columns
)

print("Total rows:", len(model_features_df))
print("Rows with all Version 1 features:", len(complete_feature_rows))
print(
    "Complete-feature coverage: {:.2f}%".format(
        len(complete_feature_rows)
        / len(model_features_df)
        * 100
    )
)

Total rows: 301275
Rows with all Version 1 features: 289412
Complete-feature coverage: 96.06%


In [138]:
full_feature_panel_path = (
    PROCESSED_DATA_DIR
    / "model_features_full_v1.parquet"
)

model_features_df.to_parquet(
    full_feature_panel_path,
    index=False,
)

print("Saved full Version 1 feature panel to:")
print(full_feature_panel_path)

Saved full Version 1 feature panel to:
/Users/kushagr/Desktop/astra-assignment/outputs/processed_data/model_features_full_v1.parquet


### Chronological Train, Validation and Test Splits

In [139]:
split_df = model_features_df.copy()

split_df = split_df.sort_values(["pred_date", "symbol"]).reset_index(drop=True)

print("Starting rows:", len(split_df))
print("Date range:", split_df["pred_date"].min(), "to", split_df["pred_date"].max())

Starting rows: 301275
Date range: 2020-06-01 00:00:00 to 2026-06-29 00:00:00


In [140]:
VALID_PERIOD_START = pd.Timestamp("2024-04-01")
TEST_PERIOD_START = pd.Timestamp("2025-05-01")
EMBARGO_SESSIONS = 5

print("Validation period begins around:", VALID_PERIOD_START.date())
print("Test period begins around:", TEST_PERIOD_START.date())
print("Embargo sessions:", EMBARGO_SESSIONS)

Validation period begins around: 2024-04-01
Test period begins around: 2025-05-01
Embargo sessions: 5


In [141]:
available_pred_dates = pd.Index(split_df["pred_date"].dropna().sort_values().unique())

print("Number of prediction dates:", len(available_pred_dates))
print("First date:", available_pred_dates.min())
print("Last date:", available_pred_dates.max())

Number of prediction dates: 1510
First date: 2020-06-01 00:00:00
Last date: 2026-06-29 00:00:00


In [142]:
train_valid_embargo_dates = available_pred_dates[available_pred_dates >= VALID_PERIOD_START][:EMBARGO_SESSIONS]

print("Train-validation embargo dates:")

for date in train_valid_embargo_dates:
    print(pd.Timestamp(date).date())

Train-validation embargo dates:
2024-04-01
2024-04-02
2024-04-03
2024-04-04
2024-04-05


In [143]:
actual_valid_start = available_pred_dates[available_pred_dates > train_valid_embargo_dates[-1]][0]

print("Actual validation start:", pd.Timestamp(actual_valid_start).date())

Actual validation start: 2024-04-08


In [144]:
valid_test_embargo_dates = available_pred_dates[available_pred_dates >= TEST_PERIOD_START][:EMBARGO_SESSIONS]

print("Validation-test embargo dates:")

for date in valid_test_embargo_dates:
    print(pd.Timestamp(date).date())

Validation-test embargo dates:
2025-05-02
2025-05-05
2025-05-06
2025-05-07
2025-05-08


In [145]:
actual_test_start = available_pred_dates[available_pred_dates > valid_test_embargo_dates[-1]][0]

print("Actual test start:", pd.Timestamp(actual_test_start).date())

Actual test start: 2025-05-09


In [146]:
split_df["split"] = "unused"

train_mask = split_df["pred_date"] < VALID_PERIOD_START

train_valid_embargo_mask = split_df["pred_date"].isin(train_valid_embargo_dates)

valid_mask = ((split_df["pred_date"] >= actual_valid_start)& (split_df["pred_date"] < TEST_PERIOD_START))

valid_test_embargo_mask = split_df["pred_date"].isin(valid_test_embargo_dates)

test_mask = split_df["pred_date"] >= actual_test_start

split_df.loc[train_mask, "split"] = "train"
split_df.loc[train_valid_embargo_mask, "split"] = "embargo"
split_df.loc[valid_mask, "split"] = "valid"
split_df.loc[valid_test_embargo_mask, "split"] = "embargo"
split_df.loc[test_mask, "split"] = "test"

split_df["split"].value_counts()

split
train      186555
test        58656
valid       54009
embargo      2055
Name: count, dtype: int64

In [147]:
split_date_summary = split_df.groupby("split").agg(
    start_date=("pred_date", "min"),
    end_date=("pred_date", "max"),
    rows=("pred_date", "size"),
    dates=("pred_date", "nunique"),
    symbols=("symbol", "nunique"),
)

split_date_summary

,start_date,end_date,rows,dates,symbols
split,,,,,
embargo,2024-04-01,2025-05-08,2055,10,208
test,2025-05-09,2026-06-29,58656,282,208
train,2020-06-01,2024-03-28,186555,955,203
valid,2024-04-08,2025-04-30,54009,263,208


In [148]:
unused_rows = split_df.loc[split_df["split"] == "unused",["pred_date", "symbol"]]

print("Unused rows:", len(unused_rows))

unused_rows.head()

Unused rows: 0


,pred_date,symbol


In [149]:
train_end = split_df.loc[split_df["split"] == "train","pred_date"].max()

valid_start = split_df.loc[split_df["split"] == "valid","pred_date"].min()

valid_end = split_df.loc[split_df["split"] == "valid","pred_date"].max()

test_start = split_df.loc[split_df["split"] == "test","pred_date"].min()

print("Train end:", train_end)
print("Validation start:", valid_start)
print("Validation end:", valid_end)
print("Test start:", test_start)

assert train_end < valid_start
assert valid_end < test_start

print("Chronological ordering confirmed.")

Train end: 2024-03-28 00:00:00
Validation start: 2024-04-08 00:00:00
Validation end: 2025-04-30 00:00:00
Test start: 2025-05-09 00:00:00
Chronological ordering confirmed.


In [150]:
print("Train-validation embargo sessions:",split_df.loc[split_df["pred_date"].isin(train_valid_embargo_dates),"pred_date"].nunique())

print("Validation-test embargo sessions:",split_df.loc[split_df["pred_date"].isin(valid_test_embargo_dates),"pred_date"].nunique())

assert len(train_valid_embargo_dates) == EMBARGO_SESSIONS
assert len(valid_test_embargo_dates) == EMBARGO_SESSIONS

print("Embargo checks passed.")


Train-validation embargo sessions: 5
Validation-test embargo sessions: 5
Embargo checks passed.


In [151]:
model_split_df = split_df.copy()

print("Completed split panel shape:", model_split_df.shape)

Completed split panel shape: (301275, 45)


In [152]:
feature_coverage_by_split = (model_split_df.groupby("split")[all_feature_columns].apply(lambda frame: frame.notna().mean() * 100))

feature_coverage_by_split.T

split,embargo,test,train,valid
lagged_overnight_return_1d,100.0000,100.0000,99.8912,99.9907
return_1d,100.0000,100.0000,99.8912,99.9907
return_5d,100.0000,100.0000,99.4559,99.9537
return_20d,100.0000,100.0000,97.8237,99.8148
daily_volatility_20d,100.0000,100.0000,97.8237,99.8148
overnight_std_20d,100.0000,100.0000,97.8237,99.8148
daily_volatility_5d,100.0000,100.0000,99.4559,99.9537
gap_mean_20d,100.0000,100.0000,97.8237,99.8148
gap_positive_fraction_20d,100.0000,100.0000,97.8237,99.8148
volume_zscore_20d,100.0000,100.0000,97.8237,99.8148


In [153]:
complete_row_summary = []

for split_name in ["train", "valid", "test"]:
    split_subset = model_split_df.loc[
        model_split_df["split"] == split_name
    ]

    complete_rows = split_subset[
        all_feature_columns
    ].notna().all(axis=1).sum()

    complete_row_summary.append(
        {
            "split": split_name,
            "total_rows": len(split_subset),
            "complete_rows": complete_rows,
            "complete_pct": complete_rows / len(split_subset) * 100,
        }
    )

complete_row_summary_df = pd.DataFrame(complete_row_summary)

complete_row_summary_df

,split,total_rows,complete_rows,complete_pct
0,train,186555,175610,94.1331
1,valid,54009,53478,99.0168
2,test,58656,58286,99.3692


In [154]:
split_panel_path = (PROCESSED_DATA_DIR/ "model_features_with_splits_v1.parquet")

model_split_df.to_parquet(split_panel_path,index=False,)

print("Saved split feature panel to:")
print(split_panel_path)

Saved split feature panel to:
/Users/kushagr/Desktop/astra-assignment/outputs/processed_data/model_features_with_splits_v1.parquet


### ML Dataset Preparation

Prepare the final modelling datasets.

Steps:

1. Define metadata columns
2. Define target columns
3. Define feature columns
4. Remove leakage columns
5. Drop warm-up rows
6. Create train / validation / test datasets
7. Create X and y matrices

In [155]:
METADATA_COLUMNS = ["symbol","pred_date","target_date","split",]

In [156]:
TARGET_COLUMNS = [
    "actual_return_pct",
    "actual_direction",
    "actual_magnitude_pct",
]

In [157]:
LEAKAGE_COLUMNS = [
    "next_open",
    "actual_return_pct",
    "actual_direction",
    "actual_magnitude_pct",
    "target_date",
    "split",
]

In [158]:
FEATURE_COLUMNS = sorted(list(set(all_feature_columns)| {"symbol"}))

print("Number of model features:", len(FEATURE_COLUMNS))

FEATURE_COLUMNS[:10]

Number of model features: 26


['aggregate_universe_volatility',
 'amihud_illiquidity',
 'calendar_gap_days',
 'close_auction_return_concentration',
 'close_auction_volume_concentration',
 'close_vwap_deviation_pct',
 'cross_sectional_return_dispersion',
 'daily_volatility_20d',
 'daily_volatility_5d',
 'day_of_week']

In [159]:
leaking_features = sorted(
    set(FEATURE_COLUMNS)
    & set(LEAKAGE_COLUMNS)
)

assert len(leaking_features) == 0

print("No leakage columns found in feature list.")

No leakage columns found in feature list.


In [160]:
ml_df = model_split_df.copy()

print("Rows:", len(ml_df))
print("Columns:", len(ml_df.columns))

Rows: 301275
Columns: 45


In [161]:
initial_rows = len(ml_df)

ml_df = ml_df.dropna(subset=daily_feature_columns).reset_index(drop=True)

print("Dropped rows:", initial_rows - len(ml_df))
print("Remaining rows:", len(ml_df))

Dropped rows: 4160
Remaining rows: 297115


In [162]:
ml_df["symbol"] = (ml_df["symbol"].astype("category"))

ml_df["symbol"].dtype

CategoricalDtype(categories=['360ONE', 'ABB', 'ABCAPITAL', 'ADANIENSOL', 'ADANIENT',
                  'ADANIGREEN', 'ADANIPORTS', 'ADANIPOWER', 'ALKEM', 'AMBER',
                  ...
                  'UNOMINDA', 'UPL', 'VBL', 'VEDL', 'VMM', 'VOLTAS',
                  'WAAREEENER', 'WIPRO', 'YESBANK', 'ZYDUSLIFE'],
, ordered=False, categories_dtype=object)

In [163]:
train_df = ml_df.loc[ml_df["split"] == "train"].copy()

valid_df = ml_df.loc[ml_df["split"] == "valid"].copy()

test_df = ml_df.loc[ml_df["split"] == "test"].copy()

print(len(train_df), len(valid_df), len(test_df))

182495 53909 58656


In [164]:
train_metadata = train_df[METADATA_COLUMNS].reset_index(drop=True)

valid_metadata = valid_df[METADATA_COLUMNS].reset_index(drop=True)

test_metadata = test_df[METADATA_COLUMNS].reset_index(drop=True)

In [165]:
X_train = train_df[FEATURE_COLUMNS].copy()

X_valid = valid_df[FEATURE_COLUMNS].copy()

X_test = test_df[FEATURE_COLUMNS].copy()

print(X_train.shape)
print(X_valid.shape)
print(X_test.shape)

(182495, 26)
(53909, 26)
(58656, 26)


In [166]:
y_direction_train = train_df["actual_direction"]

y_direction_valid = valid_df["actual_direction"]

y_direction_test = test_df["actual_direction"]

In [167]:
y_magnitude_train = np.log1p(train_df["actual_magnitude_pct"])

y_magnitude_valid = np.log1p(valid_df["actual_magnitude_pct"])

y_magnitude_test = np.log1p(test_df["actual_magnitude_pct"])

In [168]:
print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))
print("Test rows:", len(X_test))

assert len(X_train) == len(y_direction_train)
assert len(X_valid) == len(y_direction_valid)
assert len(X_test) == len(y_direction_test)

assert len(X_train) == len(y_magnitude_train)

print("ML dataset preparation complete.")

Training rows: 182495
Validation rows: 53909
Test rows: 58656
ML dataset preparation complete.


### Baseline Direction Model

Train the first pooled LightGBM classifier to predict the direction of the next overnight return.

Target:
- actual_direction

Output:
- predicted direction
- predicted probability

In [169]:
import lightgbm as lgb

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

In [170]:
y_train_binary = (y_direction_train == 1).astype(int)

y_valid_binary = (y_direction_valid == 1).astype(int)

y_test_binary = (y_direction_test == 1).astype(int)

In [171]:
categorical_features = [
    "symbol",
]

In [172]:
train_dataset = lgb.Dataset(
    X_train,
    label=y_train_binary,
    categorical_feature=categorical_features,
    free_raw_data=False,
)

valid_dataset = lgb.Dataset(
    X_valid,
    label=y_valid_binary,
    categorical_feature=categorical_features,
    free_raw_data=False,
)

In [173]:
direction_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "seed": 42,
    "verbosity": -1,
}

In [174]:
direction_model = lgb.train(
    params=direction_params,
    train_set=train_dataset,
    valid_sets=[train_dataset, valid_dataset],
    valid_names=["train", "valid"],
    num_boost_round=500,
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50),
    ],
)

Training until validation scores don't improve for 50 rounds
[50]	train's binary_logloss: 0.513628	valid's binary_logloss: 0.611494
Early stopping, best iteration is:
[39]	train's binary_logloss: 0.522065	valid's binary_logloss: 0.611009


In [175]:
valid_probability = direction_model.predict(X_valid)

test_probability = direction_model.predict(X_test)

In [176]:
valid_prediction = (valid_probability >= 0.5).astype(int)

test_prediction = (test_probability >= 0.5).astype(int)

In [177]:
valid_prediction = np.where(valid_prediction == 1,1,-1,)

test_prediction = np.where(test_prediction == 1,1,-1,)

In [178]:
validation_accuracy = accuracy_score(y_direction_valid,valid_prediction,)

print(f"Validation Accuracy: {validation_accuracy:.4f}")

Validation Accuracy: 0.6879


In [179]:
confusion_matrix(y_direction_valid,valid_prediction,)

array([[  503, 16182],
       [  644, 36580]])

In [180]:
print(classification_report(y_direction_valid,valid_prediction,))

              precision    recall  f1-score   support

          -1       0.44      0.03      0.06     16685
           1       0.69      0.98      0.81     37224

    accuracy                           0.69     53909
   macro avg       0.57      0.51      0.43     53909
weighted avg       0.61      0.69      0.58     53909



In [181]:
direction_feature_importance = (pd.DataFrame(
        {
            "feature": direction_model.feature_name(),
            "importance": direction_model.feature_importance(
                importance_type="gain"
            ),
        }
    )
    .sort_values(
        "importance",
        ascending=False,
    )
)

direction_feature_importance.head(20)

,feature,importance
14,market_breadth,"52,674.8937"
0,aggregate_universe_volatility,"41,411.0023"
6,cross_sectional_return_dispersion,"32,537.9051"
23,symbol,"16,477.6988"
9,day_of_week,"14,152.2505"
11,gap_positive_fraction_20d,"9,474.3055"
2,calendar_gap_days,"6,249.2677"
5,close_vwap_deviation_pct,"3,278.4861"
10,gap_mean_20d,"1,755.0047"
3,close_auction_return_concentration,979.8148


In [182]:
# print("="*80)
# print("VALIDATION ACCURACY")
# print("="*80)
# print(validation_accuracy)

# print("\n")
# print("="*80)
# print("CONFUSION MATRIX")
# print("="*80)
# print(confusion_matrix(y_direction_valid, valid_prediction))

# print("\n")
# print("="*80)
# print("CLASSIFICATION REPORT")
# print("="*80)
# print(classification_report(y_direction_valid, valid_prediction, digits=4))

# print("\n")
# print("="*80)
# print("TOP 20 FEATURES")
# print("="*80)
# print(direction_feature_importance.head(20).to_string(index=False))

# print("\n")
# print("="*80)
# print("PROBABILITY SUMMARY")
# print("="*80)
# print(pd.Series(valid_probability).describe())

#### Direction Feature Engineering — Version 3

Version 3 adds two feature families:

1. Daily and cross-sectional features derived from the existing daily panel.
2. Minute-derived daily features computed from raw one-minute bars.

The minute features are processed once and saved separately so that model experimentation
does not require rereading all 208 minute parquet files.

In [183]:
V3_MINUTE_FEATURES_PATH = (
    PROCESSED_DATA_DIR / "direction_v3_minute_features.parquet"
)

V3_DAILY_FEATURES_PATH = (
    PROCESSED_DATA_DIR / "direction_v3_daily_features.parquet"
)

V3_MODEL_PANEL_PATH = (
    PROCESSED_DATA_DIR / "direction_v3_model_panel.parquet"
)

REBUILD_V3_MINUTE_FEATURES = False

print("Minute feature cache:", V3_MINUTE_FEATURES_PATH)
print("Daily feature panel:", V3_DAILY_FEATURES_PATH)
print("Final V3 model panel:", V3_MODEL_PANEL_PATH)
print("Rebuild minute features:", REBUILD_V3_MINUTE_FEATURES)

Minute feature cache: /Users/kushagr/Desktop/astra-assignment/outputs/processed_data/direction_v3_minute_features.parquet
Daily feature panel: /Users/kushagr/Desktop/astra-assignment/outputs/processed_data/direction_v3_daily_features.parquet
Final V3 model panel: /Users/kushagr/Desktop/astra-assignment/outputs/processed_data/direction_v3_model_panel.parquet
Rebuild minute features: False


#### V3 Daily and Cross-Sectional Features

These features are inexpensive to calculate and therefore may be regenerated whenever
the V3 dataset is rebuilt.

All rolling statistics use current or historical information available by the close of
prediction date T. Historical overnight-return features are shifted before rolling because
the current row's overnight target is realised only at the next session's open.

In [185]:
direction_v3_daily_df = daily_features_df.copy()

direction_v3_daily_df = (
    direction_v3_daily_df
    .sort_values(["symbol", "pred_date"])
    .reset_index(drop=True)
)

print("Starting V3 daily shape:", direction_v3_daily_df.shape)
print("Symbols:", direction_v3_daily_df["symbol"].nunique())
print(
    "Date range:",
    direction_v3_daily_df["pred_date"].min(),
    "to",
    direction_v3_daily_df["pred_date"].max(),
)

Starting V3 daily shape: (301275, 29)
Symbols: 208
Date range: 2020-06-01 00:00:00 to 2026-06-29 00:00:00


In [187]:
existing_market_columns = [
    column
    for column in [
        "market_breadth",
        "cross_sectional_return_dispersion",
        "aggregate_universe_volatility",
    ]
    if column in direction_v3_daily_df.columns
]

print("Market columns already present:", existing_market_columns)

Market columns already present: []


In [188]:
market_daily_features = (
    direction_v3_daily_df
    .groupby("pred_date", as_index=False)
    .agg(
        universe_return_1d=(
            "return_1d",
            "mean",
        ),
        market_breadth=(
            "return_1d",
            lambda series: series.gt(0).mean(),
        ),
        cross_sectional_return_dispersion=(
            "return_1d",
            "std",
        ),
        aggregate_universe_volatility=(
            "daily_volatility_20d",
            "mean",
        ),
    )
    .sort_values("pred_date")
    .reset_index(drop=True)
)

market_daily_features.head()

,pred_date,universe_return_1d,market_breadth,cross_sectional_return_dispersion,aggregate_universe_volatility
0,2020-06-01,NaN,0.0000,NaN,NaN
1,2020-06-02,1.7718,0.7446,2.9029,NaN
2,2020-06-03,1.0758,0.5978,2.6739,NaN
3,2020-06-04,0.0977,0.4348,2.7574,NaN
4,2020-06-05,2.6821,0.8043,3.4775,NaN


In [189]:
direction_v3_daily_df = direction_v3_daily_df.merge(
    market_daily_features,
    on="pred_date",
    how="left",
    validate="many_to_one",
)

print("Shape after market-feature merge:", direction_v3_daily_df.shape)

direction_v3_daily_df[
    [
        "pred_date",
        "symbol",
        "return_1d",
        "universe_return_1d",
        "market_breadth",
        "cross_sectional_return_dispersion",
        "aggregate_universe_volatility",
    ]
].head()

Shape after market-feature merge: (301275, 33)


,pred_date,symbol,return_1d,universe_return_1d,market_breadth,cross_sectional_return_dispersion,aggregate_universe_volatility
0,2020-06-01,360ONE,NaN,NaN,0.0000,NaN,NaN
1,2020-06-02,360ONE,2.4629,1.7718,0.7446,2.9029,NaN
2,2020-06-03,360ONE,1.3577,1.0758,0.5978,2.6739,NaN
3,2020-06-04,360ONE,5.3360,0.0977,0.4348,2.7574,NaN
4,2020-06-05,360ONE,4.2944,2.6821,0.8043,3.4775,NaN


In [190]:
assert direction_v3_daily_df[
    "market_breadth"
].dropna().between(0, 1).all()

assert (
    direction_v3_daily_df[
        "cross_sectional_return_dispersion"
    ].dropna() >= 0
).all()

assert (
    direction_v3_daily_df[
        "aggregate_universe_volatility"
    ].dropna() >= 0
).all()

market_feature_columns = [
    "universe_return_1d",
    "market_breadth",
    "cross_sectional_return_dispersion",
    "aggregate_universe_volatility",
]

print(
    direction_v3_daily_df[
        market_feature_columns
    ].isna().sum()
)

print("Market feature checks passed.")

universe_return_1d                    184
market_breadth                          0
cross_sectional_return_dispersion     184
aggregate_universe_volatility        3680
dtype: int64
Market feature checks passed.


#### V3 Cross-Sectional and Regime Features

The following features measure whether a stock's current movement is unusual relative
to the rest of the universe and whether the current market environment differs from its
recent history.

All rolling regime statistics use only the current and previous prediction dates.

In [192]:
direction_v3_daily_df["idiosyncratic_return_1d"] = (
    direction_v3_daily_df["return_1d"]
    - direction_v3_daily_df["universe_return_1d"]
)

In [193]:
direction_v3_daily_df["relative_return_rank_1d"] = (
    direction_v3_daily_df
    .groupby("pred_date")["return_1d"]
    .rank(
        method="average",
        pct=True,
    )
)

In [208]:
direction_v3_daily_df["relative_momentum_rank_5d"] = (
    direction_v3_daily_df
    .groupby("pred_date")["return_5d"]
    .rank(
        method="average",
        pct=True,
    )
)

In [209]:
market_regime_df = (
    market_daily_features[
        [
            "pred_date",
            "market_breadth",
            "cross_sectional_return_dispersion",
        ]
    ]
    .sort_values("pred_date")
    .reset_index(drop=True)
)

# ---------------- Breadth regime ---------------- #

market_regime_df["breadth_mean_20d_lagged"] = (
    market_regime_df["market_breadth"]
    .shift(1)
    .rolling(window=20, min_periods=20)
    .mean()
)

market_regime_df["breadth_std_20d_lagged"] = (
    market_regime_df["market_breadth"]
    .shift(1)
    .rolling(window=20, min_periods=20)
    .std()
)

breadth_std = (
    market_regime_df["breadth_std_20d_lagged"]
    .replace(0, np.nan)
)

market_regime_df["breadth_shock_zscore"] = (
    market_regime_df["market_breadth"]
    - market_regime_df["breadth_mean_20d_lagged"]
) / breadth_std

# ---------------- Dispersion regime ---------------- #

market_regime_df["dispersion_mean_20d_lagged"] = (
    market_regime_df["cross_sectional_return_dispersion"]
    .shift(1)
    .rolling(window=20, min_periods=20)
    .mean()
)

market_regime_df["dispersion_std_20d_lagged"] = (
    market_regime_df["cross_sectional_return_dispersion"]
    .shift(1)
    .rolling(window=20, min_periods=20)
    .std()
)

dispersion_std = (
    market_regime_df["dispersion_std_20d_lagged"]
    .replace(0, np.nan)
)

market_regime_df["dispersion_regime_zscore"] = (
    market_regime_df["cross_sectional_return_dispersion"]
    - market_regime_df["dispersion_mean_20d_lagged"]
) / dispersion_std

market_regime_df.head()

,pred_date,market_breadth,cross_sectional_return_dispersion,breadth_mean_20d_lagged,breadth_std_20d_lagged,breadth_shock_zscore,dispersion_mean_20d_lagged,dispersion_std_20d_lagged,dispersion_regime_zscore
0,2020-06-01,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-06-02,0.7446,2.9029,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-06-03,0.5978,2.6739,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-06-04,0.4348,2.7574,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-06-05,0.8043,3.4775,NaN,NaN,NaN,NaN,NaN,NaN


In [210]:
direction_v3_daily_df = direction_v3_daily_df.merge(
    market_regime_df[
        [
            "pred_date",
            "breadth_shock_zscore",
            "dispersion_regime_zscore",
        ]
    ],
    on="pred_date",
    how="left",
    validate="many_to_one",
)

print("Shape after regime-feature merge:", direction_v3_daily_df.shape)

Shape after regime-feature merge: (301275, 42)


In [211]:
v3_cross_sectional_feature_columns = [
    "idiosyncratic_return_1d",
    "relative_return_rank_1d",
    "relative_momentum_rank_5d",
    "breadth_shock_zscore",
    "dispersion_regime_zscore",
]

assert direction_v3_daily_df[
    "relative_return_rank_1d"
].dropna().between(0, 1).all()

assert direction_v3_daily_df[
    "relative_momentum_rank_5d"
].dropna().between(0, 1).all()

assert not np.isinf(
    direction_v3_daily_df[
        v3_cross_sectional_feature_columns
    ].to_numpy(dtype=float)
).any()

direction_v3_daily_df[
    v3_cross_sectional_feature_columns
].describe().T

,count,mean,std,min,25%,50%,75%,max
idiosyncratic_return_1d,"301,067.0000",0.0000,1.9723,-31.6583,-1.0419,-0.1425,0.8720,84.4301
relative_return_rank_1d,"301,067.0000",0.5025,0.2887,0.0048,0.2525,0.5025,0.7525,1.0000
relative_momentum_rank_5d,"300,235.0000",0.5025,0.2887,0.0048,0.2525,0.5025,0.7525,1.0000
breadth_shock_zscore,"297,595.0000",-0.0281,1.0692,-4.1298,-0.7511,0.0562,0.7577,2.8347
dispersion_regime_zscore,"297,411.0000",0.0342,1.2999,-3.9525,-0.7568,-0.1265,0.5921,13.5062


In [212]:
direction_v3_daily_df[
    v3_cross_sectional_feature_columns
].isna().sum()

idiosyncratic_return_1d       208
relative_return_rank_1d       208
relative_momentum_rank_5d    1040
breadth_shock_zscore         3680
dispersion_regime_zscore     3864
dtype: int64

In [213]:
v3_na_diagnostics = pd.DataFrame({
    "nan_count": direction_v3_daily_df[
        v3_cross_sectional_feature_columns
    ].isna().sum(),
    "nan_pct": (
        direction_v3_daily_df[
            v3_cross_sectional_feature_columns
        ].isna().mean() * 100
    ),
})

v3_na_diagnostics

,nan_count,nan_pct
idiosyncratic_return_1d,208,0.0690
relative_return_rank_1d,208,0.0690
relative_momentum_rank_5d,1040,0.3452
breadth_shock_zscore,3680,1.2215
dispersion_regime_zscore,3864,1.2825


In [214]:
for column in [
    "breadth_shock_zscore",
    "dispersion_regime_zscore",
]:
    print(f"\n{column}")
    print(
        direction_v3_daily_df.loc[
            direction_v3_daily_df[column].isna(),
            "pred_date",
        ].drop_duplicates().sort_values().head(25).tolist()
    )


breadth_shock_zscore
[Timestamp('2020-06-01 00:00:00'), Timestamp('2020-06-02 00:00:00'), Timestamp('2020-06-03 00:00:00'), Timestamp('2020-06-04 00:00:00'), Timestamp('2020-06-05 00:00:00'), Timestamp('2020-06-08 00:00:00'), Timestamp('2020-06-09 00:00:00'), Timestamp('2020-06-10 00:00:00'), Timestamp('2020-06-11 00:00:00'), Timestamp('2020-06-12 00:00:00'), Timestamp('2020-06-15 00:00:00'), Timestamp('2020-06-16 00:00:00'), Timestamp('2020-06-17 00:00:00'), Timestamp('2020-06-18 00:00:00'), Timestamp('2020-06-19 00:00:00'), Timestamp('2020-06-22 00:00:00'), Timestamp('2020-06-23 00:00:00'), Timestamp('2020-06-24 00:00:00'), Timestamp('2020-06-25 00:00:00'), Timestamp('2020-06-26 00:00:00')]

dispersion_regime_zscore
[Timestamp('2020-06-01 00:00:00'), Timestamp('2020-06-02 00:00:00'), Timestamp('2020-06-03 00:00:00'), Timestamp('2020-06-04 00:00:00'), Timestamp('2020-06-05 00:00:00'), Timestamp('2020-06-08 00:00:00'), Timestamp('2020-06-09 00:00:00'), Timestamp('2020-06-10 00:00:00')

#### V3 Gap Behaviour Features

These features describe whether the current overnight gap is unusually large relative
to the stock's own history and whether the stock reverses or continues that gap during
the trading session.

Only information available by the close of prediction date T is used.

In [215]:
required_gap_columns = [
    "symbol",
    "pred_date",
    "open",
    "close",
]

missing_gap_columns = [
    column
    for column in required_gap_columns
    if column not in direction_v3_daily_df.columns
]

assert not missing_gap_columns, (
    f"Missing required columns: {missing_gap_columns}"
)

direction_v3_daily_df = (
    direction_v3_daily_df
    .sort_values(["symbol", "pred_date"])
    .reset_index(drop=True)
)

In [216]:
direction_v3_daily_df["current_gap_pct"] = (
    direction_v3_daily_df["open"]
    / direction_v3_daily_df.groupby("symbol")["close"].shift(1)
    - 1
) * 100

direction_v3_daily_df["intraday_return_pct"] = (
    direction_v3_daily_df["close"]
    / direction_v3_daily_df["open"]
    - 1
) * 100

In [217]:
symbol_groups = direction_v3_daily_df.groupby(
    "symbol",
    group_keys=False,
)

direction_v3_daily_df["gap_mean_20d_lagged"] = (
    symbol_groups["current_gap_pct"]
    .transform(
        lambda series: (
            series.shift(1)
            .rolling(window=20, min_periods=20)
            .mean()
        )
    )
)

direction_v3_daily_df["gap_std_20d_lagged"] = (
    symbol_groups["current_gap_pct"]
    .transform(
        lambda series: (
            series.shift(1)
            .rolling(window=20, min_periods=20)
            .std()
        )
    )
)

In [218]:
gap_std_safe = (
    direction_v3_daily_df["gap_std_20d_lagged"]
    .replace(0, np.nan)
)

direction_v3_daily_df["standardized_current_gap"] = (
    direction_v3_daily_df["current_gap_pct"]
    - direction_v3_daily_df["gap_mean_20d_lagged"]
) / gap_std_safe

direction_v3_daily_df["gap_intraday_reversal"] = (
    -np.sign(direction_v3_daily_df["current_gap_pct"])
    * direction_v3_daily_df["intraday_return_pct"]
)

In [219]:
v3_gap_feature_columns = [
    "standardized_current_gap",
    "gap_intraday_reversal",
]

assert not np.isinf(
    direction_v3_daily_df[
        v3_gap_feature_columns
    ].to_numpy(dtype=float)
).any()

print(
    direction_v3_daily_df[
        v3_gap_feature_columns
    ].isna().sum()
)

direction_v3_daily_df[
    [
        "symbol",
        "pred_date",
        "current_gap_pct",
        "intraday_return_pct",
        "standardized_current_gap",
        "gap_intraday_reversal",
    ]
].describe().T

standardized_current_gap    4368
gap_intraday_reversal        208
dtype: int64


,count,mean,min,25%,50%,75%,max,std
pred_date,301275,2023-07-02 13:44:11.399551744,2020-06-01 00:00:00,2021-12-30 00:00:00,2023-07-13 00:00:00,2025-01-08 00:00:00,2026-06-29 00:00:00,NaN
current_gap_pct,"301,067.0000",0.2144,-34.7406,-0.0672,0.1934,0.5803,63.8095,1.0074
intraday_return_pct,"301,275.0000",-0.0867,-33.3611,-1.1948,-0.1594,0.9211,22.0630,2.0962
standardized_current_gap,"296,907.0000",-0.0143,-47.5290,-0.5027,-0.0004,0.5362,58.8886,1.4068
gap_intraday_reversal,"301,067.0000",0.1164,-23.8721,-0.7975,0.0000,1.1059,33.3611,2.0237


In [221]:
direction_v3_daily_df[
    v3_gap_feature_columns
].isna().sum()

standardized_current_gap    4368
gap_intraday_reversal        208
dtype: int64

In [222]:
direction_v3_daily_df.to_parquet(
    V3_DAILY_FEATURES_PATH,
    index=False,
)

print("Saved:", V3_DAILY_FEATURES_PATH)
print(direction_v3_daily_df.shape)

Saved: /Users/kushagr/Desktop/astra-assignment/outputs/processed_data/direction_v3_daily_features.parquet
(301275, 48)


#### V3 Minute-Derived Daily Features

The raw one-minute files are processed once to create one row per symbol and trading date.

The features capture:

- Intraday price path
- Realised volatility and asymmetry
- Early-versus-late session behaviour
- Volume concentration and closing pressure
- Closing-session trend
- Data coverage and session completeness

The resulting minute feature dataset is saved separately before being merged with the
V3 daily feature panel.

In [223]:
V3_MINUTE_FEATURE_COLUMNS = [
    "intraday_path_efficiency",
    "intraday_realized_volatility",
    "intraday_realized_skewness",
    "intraday_up_minute_fraction",
    "late_session_volatility_share_60m",
    "early_session_volatility_share_60m",
    "largest_minute_move_share",
    "signed_closing_volume_pressure",
    "closing_volume_share_60m",
    "closing_return_60m",
    "closing_trend_slope",
    "intraday_high_low_range_pct",
    "close_location_in_range",
    "volume_concentration_hhi",
    "minute_bar_coverage",
]

In [224]:
def create_direction_v3_minute_features(
    minute_df: pd.DataFrame,
    symbol: str,
    expected_bars_per_session: int = 375,
) -> pd.DataFrame:
    """
    Convert one symbol's raw one-minute bars into one row per trading session.

    All features use information available up to the close of session T.
    """

    required_columns = {
        "timestamp",
        "open",
        "high",
        "low",
        "close",
        "volume",
    }

    missing_columns = required_columns.difference(minute_df.columns)

    if missing_columns:
        raise ValueError(
            f"{symbol}: missing minute columns: {sorted(missing_columns)}"
        )

    df = minute_df[
        [
            "timestamp",
            "open",
            "high",
            "low",
            "close",
            "volume",
        ]
    ].copy()

    df["timestamp"] = pd.to_datetime(df["timestamp"])

    numeric_columns = [
        "open",
        "high",
        "low",
        "close",
        "volume",
    ]

    for column in numeric_columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )

    df = (
        df.dropna(
            subset=[
                "timestamp",
                "open",
                "high",
                "low",
                "close",
            ]
        )
        .sort_values("timestamp")
        .drop_duplicates("timestamp", keep="last")
        .reset_index(drop=True)
    )

    df = df[
        (df["open"] > 0)
        & (df["high"] > 0)
        & (df["low"] > 0)
        & (df["close"] > 0)
    ].copy()

    df["volume"] = df["volume"].fillna(0).clip(lower=0)

    df["pred_date"] = df["timestamp"].dt.normalize()

    # Remove bars outside the regular session.
    minute_of_day = (
        df["timestamp"].dt.hour * 60
        + df["timestamp"].dt.minute
    )

    session_start = 9 * 60 + 15
    session_end = 15 * 60 + 29

    df = df[
        minute_of_day.between(
            session_start,
            session_end,
        )
    ].copy()

    if df.empty:
        return pd.DataFrame(
            columns=[
                "symbol",
                "pred_date",
                *V3_MINUTE_FEATURE_COLUMNS,
            ]
        )

    output_rows = []

    for pred_date, day in df.groupby(
        "pred_date",
        sort=True,
    ):
        day = (
            day.sort_values("timestamp")
            .reset_index(drop=True)
        )

        n_bars = len(day)

        if n_bars < 2:
            continue

        close = day["close"].to_numpy(dtype=float)
        open_price = day["open"].to_numpy(dtype=float)
        high = day["high"].to_numpy(dtype=float)
        low = day["low"].to_numpy(dtype=float)
        volume = day["volume"].to_numpy(dtype=float)

        # Log returns are additive and stable for path calculations.
        minute_log_returns = np.diff(np.log(close))

        finite_returns = minute_log_returns[
            np.isfinite(minute_log_returns)
        ]

        if finite_returns.size == 0:
            continue

        absolute_returns = np.abs(finite_returns)
        squared_returns = finite_returns ** 2

        total_absolute_path = absolute_returns.sum()

        session_log_move = abs(
            np.log(close[-1] / open_price[0])
        )

        if total_absolute_path > 0:
            intraday_path_efficiency = (
                session_log_move / total_absolute_path
            )
        else:
            intraday_path_efficiency = 0.0

        # Numerical and bar-construction differences can otherwise
        # produce tiny values above one.
        intraday_path_efficiency = float(
            np.clip(
                intraday_path_efficiency,
                0.0,
                1.0,
            )
        )

        intraday_realized_volatility = float(
            np.sqrt(squared_returns.sum()) * 100
        )

        if (
            finite_returns.size >= 3
            and np.std(finite_returns, ddof=1) > 0
        ):
            intraday_realized_skewness = float(
                pd.Series(finite_returns).skew()
            )
        else:
            intraday_realized_skewness = 0.0

        intraday_up_minute_fraction = float(
            np.mean(finite_returns > 0)
        )

        return_count = len(finite_returns)
        window_size = min(60, return_count)

        total_squared_return = squared_returns.sum()

        if total_squared_return > 0:
            early_session_volatility_share_60m = float(
                squared_returns[:window_size].sum()
                / total_squared_return
            )

            late_session_volatility_share_60m = float(
                squared_returns[-window_size:].sum()
                / total_squared_return
            )

            largest_minute_move_share = float(
                squared_returns.max()
                / total_squared_return
            )
        else:
            early_session_volatility_share_60m = 0.0
            late_session_volatility_share_60m = 0.0
            largest_minute_move_share = 0.0

        total_volume = volume.sum()
        closing_bar_count = min(60, n_bars)

        closing_volume = volume[-closing_bar_count:]
        closing_close = close[-closing_bar_count:]

        if total_volume > 0:
            closing_volume_share_60m = float(
                closing_volume.sum() / total_volume
            )

            volume_weights = volume / total_volume

            volume_concentration_hhi = float(
                np.square(volume_weights).sum()
            )
        else:
            closing_volume_share_60m = 0.0
            volume_concentration_hhi = 0.0

        # Assign each closing bar's volume the sign of its close-to-close move.
        closing_price_changes = np.diff(
            close[-(closing_bar_count + 1):]
        )

        if closing_price_changes.size == closing_volume.size:
            closing_signs = np.sign(closing_price_changes)
            signed_closing_volume = (
                closing_signs * closing_volume
            ).sum()
        else:
            closing_signs = np.sign(
                np.diff(closing_close, prepend=closing_close[0])
            )

            signed_closing_volume = (
                closing_signs * closing_volume
            ).sum()

        if closing_volume.sum() > 0:
            signed_closing_volume_pressure = float(
                signed_closing_volume
                / closing_volume.sum()
            )
        else:
            signed_closing_volume_pressure = 0.0

        if n_bars > closing_bar_count:
            closing_reference_price = close[
                -(closing_bar_count + 1)
            ]
        else:
            closing_reference_price = open_price[0]

        closing_return_60m = float(
            (
                close[-1]
                / closing_reference_price
                - 1
            )
            * 100
        )

        if closing_bar_count >= 2:
            closing_x = np.arange(
                closing_bar_count,
                dtype=float,
            )

            closing_log_prices = np.log(closing_close)

            closing_trend_slope = float(
                np.polyfit(
                    closing_x,
                    closing_log_prices,
                    1,
                )[0]
                * 100
            )
        else:
            closing_trend_slope = 0.0

        session_high = np.max(high)
        session_low = np.min(low)

        intraday_high_low_range_pct = float(
            (
                session_high
                / session_low
                - 1
            )
            * 100
        )

        price_range = session_high - session_low

        if price_range > 0:
            close_location_in_range = float(
                (
                    close[-1] - session_low
                )
                / price_range
            )
        else:
            close_location_in_range = 0.5

        minute_bar_coverage = float(
            min(
                n_bars / expected_bars_per_session,
                1.0,
            )
        )

        output_rows.append(
            {
                "symbol": symbol,
                "pred_date": pd.Timestamp(pred_date),
                "intraday_path_efficiency": intraday_path_efficiency,
                "intraday_realized_volatility": intraday_realized_volatility,
                "intraday_realized_skewness": intraday_realized_skewness,
                "intraday_up_minute_fraction": intraday_up_minute_fraction,
                "late_session_volatility_share_60m": (
                    late_session_volatility_share_60m
                ),
                "early_session_volatility_share_60m": (
                    early_session_volatility_share_60m
                ),
                "largest_minute_move_share": largest_minute_move_share,
                "signed_closing_volume_pressure": (
                    signed_closing_volume_pressure
                ),
                "closing_volume_share_60m": closing_volume_share_60m,
                "closing_return_60m": closing_return_60m,
                "closing_trend_slope": closing_trend_slope,
                "intraday_high_low_range_pct": (
                    intraday_high_low_range_pct
                ),
                "close_location_in_range": close_location_in_range,
                "volume_concentration_hhi": volume_concentration_hhi,
                "minute_bar_coverage": minute_bar_coverage,
            }
        )

    result = pd.DataFrame(output_rows)

    if result.empty:
        return pd.DataFrame(
            columns=[
                "symbol",
                "pred_date",
                *V3_MINUTE_FEATURE_COLUMNS,
            ]
        )

    result = (
        result[
            [
                "symbol",
                "pred_date",
                *V3_MINUTE_FEATURE_COLUMNS,
            ]
        ]
        .sort_values(["symbol", "pred_date"])
        .reset_index(drop=True)
    )

    return result

In [225]:
minute_files_v3 = sorted(
    MINUTE_DATA_DIR.glob("*.parquet")
)

print("Minute files found:", len(minute_files_v3))

assert len(minute_files_v3) > 0, (
    f"No parquet files found in {MINUTE_DATA_DIR}"
)

minute_files_v3[:5]

Minute files found: 208


[PosixPath('/Users/kushagr/Desktop/astra-assignment/data/minute/360ONE.parquet'),
 PosixPath('/Users/kushagr/Desktop/astra-assignment/data/minute/ABB.parquet'),
 PosixPath('/Users/kushagr/Desktop/astra-assignment/data/minute/ABCAPITAL.parquet'),
 PosixPath('/Users/kushagr/Desktop/astra-assignment/data/minute/ADANIENSOL.parquet'),
 PosixPath('/Users/kushagr/Desktop/astra-assignment/data/minute/ADANIENT.parquet')]

In [226]:
test_minute_file = minute_files_v3[0]
test_symbol = test_minute_file.stem

test_minute_raw_df = pd.read_parquet(
    test_minute_file
)

test_minute_features_df = (
    create_direction_v3_minute_features(
        minute_df=test_minute_raw_df,
        symbol=test_symbol,
    )
)

print("Test symbol:", test_symbol)
print("Raw minute shape:", test_minute_raw_df.shape)
print(
    "Daily minute-feature shape:",
    test_minute_features_df.shape,
)

test_minute_features_df.head()

Test symbol: 360ONE
Raw minute shape: (552977, 6)
Daily minute-feature shape: (1505, 17)


,symbol,pred_date,intraday_path_efficiency,intraday_realized_volatility,intraday_realized_skewness,intraday_up_minute_fraction,late_session_volatility_share_60m,early_session_volatility_share_60m,largest_minute_move_share,signed_closing_volume_pressure,closing_volume_share_60m,closing_return_60m,closing_trend_slope,intraday_high_low_range_pct,close_location_in_range,volume_concentration_hhi,minute_bar_coverage
0,360ONE,2020-06-01,0.0671,6.7155,0.3430,0.3062,0.3872,0.2451,0.0847,-0.3100,0.1778,-4.5902,-0.0708,10.4231,0.4174,0.0584,0.9867
1,360ONE,2020-06-02,0.0009,3.9713,0.6280,0.1556,0.0853,0.2089,0.1554,0.5187,0.4931,0.0667,-0.0085,4.4998,0.6193,0.0437,0.6880
2,360ONE,2020-06-03,0.0000,4.4387,0.3020,0.2174,0.3180,0.3387,0.0634,0.1623,0.3660,0.4611,0.0065,3.9548,0.8571,0.0253,0.6773
3,360ONE,2020-06-04,0.0314,7.5344,4.0984,0.1833,0.1017,0.6539,0.3819,-0.3871,0.1552,-0.5027,0.0145,8.1967,0.4667,0.0471,0.8027
4,360ONE,2020-06-05,0.0579,4.9198,1.7470,0.1899,0.1994,0.1668,0.2356,0.1676,0.1947,0.5840,0.0194,6.9979,0.8006,0.0406,0.8453


In [227]:
assert not test_minute_features_df.empty

assert not test_minute_features_df.duplicated(
    ["symbol", "pred_date"]
).any()

bounded_zero_one_features = [
    "intraday_path_efficiency",
    "intraday_up_minute_fraction",
    "late_session_volatility_share_60m",
    "early_session_volatility_share_60m",
    "largest_minute_move_share",
    "closing_volume_share_60m",
    "close_location_in_range",
    "volume_concentration_hhi",
    "minute_bar_coverage",
]

for column in bounded_zero_one_features:
    assert test_minute_features_df[
        column
    ].dropna().between(0, 1.000001).all(), column

assert test_minute_features_df[
    "signed_closing_volume_pressure"
].dropna().between(-1.000001, 1.000001).all()

non_negative_features = [
    "intraday_realized_volatility",
    "intraday_high_low_range_pct",
]

for column in non_negative_features:
    assert (
        test_minute_features_df[column]
        .dropna()
        .ge(0)
        .all()
    ), column

assert not np.isinf(
    test_minute_features_df[
        V3_MINUTE_FEATURE_COLUMNS
    ].to_numpy(dtype=float)
).any()

print("Single-symbol minute validation passed.")

test_minute_features_df[
    V3_MINUTE_FEATURE_COLUMNS
].describe().T

Single-symbol minute validation passed.


,count,mean,std,min,25%,50%,75%,max
intraday_path_efficiency,"1,505.0000",0.0620,0.0479,0.0000,0.0243,0.0527,0.0898,0.3286
intraday_realized_volatility,"1,505.0000",2.7151,1.1552,0.8011,1.9632,2.4616,3.1588,14.1214
intraday_realized_skewness,"1,505.0000",0.0434,1.5061,-11.8740,-0.6891,0.0581,0.7355,11.8818
intraday_up_minute_fraction,"1,505.0000",0.3586,0.1034,0.0837,0.2781,0.3824,0.4439,0.5535
late_session_volatility_share_60m,"1,505.0000",0.2434,0.1596,0.0076,0.1250,0.2040,0.3211,1.0000
early_session_volatility_share_60m,"1,505.0000",0.3329,0.1555,0.0060,0.2225,0.3162,0.4271,1.0000
largest_minute_move_share,"1,505.0000",0.1323,0.0804,0.0320,0.0785,0.1117,0.1606,0.7586
signed_closing_volume_pressure,"1,505.0000",-0.0121,0.2706,-0.9773,-0.1875,-0.0136,0.1502,0.9822
closing_volume_share_60m,"1,505.0000",0.2625,0.1574,0.0070,0.1580,0.2293,0.3272,1.0000
closing_return_60m,"1,505.0000",-0.0655,1.1313,-8.3092,-0.5557,-0.0507,0.4207,10.1101


In [229]:
from time import perf_counter

minute_feature_frames = []
minute_processing_failures = []

start_time = perf_counter()

for file_number, minute_file in enumerate(
    minute_files_v3,
    start=1,
):
    symbol = minute_file.stem

    try:
        minute_raw_df = pd.read_parquet(minute_file)

        symbol_minute_features = (
            create_direction_v3_minute_features(
                minute_df=minute_raw_df,
                symbol=symbol,
            )
        )

        minute_feature_frames.append(
            symbol_minute_features
        )

    except Exception as exc:
        minute_processing_failures.append(
            {
                "symbol": symbol,
                "file": str(minute_file),
                "error": repr(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 10 == 0
        or file_number == len(minute_files_v3)
    ):
        elapsed_minutes = (
            perf_counter() - start_time
        ) / 60

        print(
            f"Processed {file_number}/{len(minute_files_v3)} "
            f"files | failures: {len(minute_processing_failures)} "
            f"| elapsed: {elapsed_minutes:.1f} minutes"
        )

elapsed_minutes = (
    perf_counter() - start_time
) / 60

print("\nMinute processing complete.")
print("Successful files:", len(minute_feature_frames))
print("Failed files:", len(minute_processing_failures))
print(f"Runtime: {elapsed_minutes:.1f} minutes")

Processed 1/208 files | failures: 0 | elapsed: 0.0 minutes
Processed 10/208 files | failures: 0 | elapsed: 0.1 minutes
Processed 20/208 files | failures: 0 | elapsed: 0.1 minutes
Processed 30/208 files | failures: 0 | elapsed: 0.2 minutes
Processed 40/208 files | failures: 0 | elapsed: 0.3 minutes
Processed 50/208 files | failures: 0 | elapsed: 0.4 minutes
Processed 60/208 files | failures: 0 | elapsed: 0.4 minutes
Processed 70/208 files | failures: 0 | elapsed: 0.5 minutes
Processed 80/208 files | failures: 0 | elapsed: 0.5 minutes
Processed 90/208 files | failures: 0 | elapsed: 0.6 minutes
Processed 100/208 files | failures: 0 | elapsed: 0.7 minutes
Processed 110/208 files | failures: 0 | elapsed: 0.7 minutes
Processed 120/208 files | failures: 0 | elapsed: 0.8 minutes
Processed 130/208 files | failures: 0 | elapsed: 0.8 minutes
Processed 140/208 files | failures: 0 | elapsed: 0.9 minutes
Processed 150/208 files | failures: 0 | elapsed: 0.9 minutes
Processed 160/208 files | failures:

In [230]:
if minute_processing_failures:
    minute_failure_df = pd.DataFrame(
        minute_processing_failures
    )

    display(minute_failure_df)
else:
    print("No minute-file processing failures.")

No minute-file processing failures.


In [231]:
direction_v3_minute_df = pd.concat(
    minute_feature_frames,
    ignore_index=True,
)

direction_v3_minute_df = (
    direction_v3_minute_df
    .sort_values(["symbol", "pred_date"])
    .reset_index(drop=True)
)

direction_v3_minute_df["pred_date"] = pd.to_datetime(
    direction_v3_minute_df["pred_date"]
)

print("Minute feature shape:", direction_v3_minute_df.shape)
print(
    "Symbols:",
    direction_v3_minute_df["symbol"].nunique(),
)
print(
    "Date range:",
    direction_v3_minute_df["pred_date"].min(),
    "to",
    direction_v3_minute_df["pred_date"].max(),
)

Minute feature shape: (300289, 17)
Symbols: 208
Date range: 2020-06-01 00:00:00 to 2026-06-29 00:00:00


In [233]:
assert (
    direction_v3_minute_df["symbol"].nunique()
    == len(minute_files_v3)
)

assert not direction_v3_minute_df.duplicated(
    ["symbol", "pred_date"]
).any()

bounded_zero_one_features = [
    "intraday_path_efficiency",
    "intraday_up_minute_fraction",
    "late_session_volatility_share_60m",
    "early_session_volatility_share_60m",
    "largest_minute_move_share",
    "closing_volume_share_60m",
    "close_location_in_range",
    "volume_concentration_hhi",
    "minute_bar_coverage",
]

for column in bounded_zero_one_features:
    assert direction_v3_minute_df[
        column
    ].dropna().between(
        0,
        1.000001,
    ).all(), column

assert direction_v3_minute_df[
    "signed_closing_volume_pressure"
].dropna().between(
    -1.000001,
    1.000001,
).all()

for column in [
    "intraday_realized_volatility",
    "intraday_high_low_range_pct",
]:
    assert direction_v3_minute_df[
        column
    ].dropna().ge(0).all(), column

assert not np.isinf(
    direction_v3_minute_df[
        V3_MINUTE_FEATURE_COLUMNS
    ].to_numpy(dtype=float)
).any()

print("Full V3 minute validation passed.")

Full V3 minute validation passed.


In [234]:
direction_v3_minute_df.to_parquet(
    V3_MINUTE_FEATURES_PATH,
    index=False,
)

print("Saved:", V3_MINUTE_FEATURES_PATH)
print("Shape:", direction_v3_minute_df.shape)

Saved: /Users/kushagr/Desktop/astra-assignment/outputs/processed_data/direction_v3_minute_features.parquet
Shape: (300289, 17)


In [235]:
direction_v3_daily_df["pred_date"] = pd.to_datetime(
    direction_v3_daily_df["pred_date"]
)

direction_v3_minute_df["pred_date"] = pd.to_datetime(
    direction_v3_minute_df["pred_date"]
)

print(
    "Daily duplicate keys:",
    direction_v3_daily_df.duplicated(
        ["symbol", "pred_date"]
    ).sum(),
)

print(
    "Minute duplicate keys:",
    direction_v3_minute_df.duplicated(
        ["symbol", "pred_date"]
    ).sum(),
)

Daily duplicate keys: 0
Minute duplicate keys: 0


In [236]:
direction_v3_model_panel = (
    direction_v3_daily_df.merge(
        direction_v3_minute_df,
        on=["symbol", "pred_date"],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)

print(
    direction_v3_model_panel["_merge"]
    .value_counts(dropna=False)
)

print(
    "Final panel shape:",
    direction_v3_model_panel.shape,
)

_merge
both          300289
left_only        986
right_only         0
Name: count, dtype: int64
Final panel shape: (301275, 64)


In [237]:
direction_v3_model_panel[
    "minute_features_available"
] = (
    direction_v3_model_panel["_merge"]
    .eq("both")
    .astype("int8")
)

direction_v3_model_panel = (
    direction_v3_model_panel
    .drop(columns="_merge")
)

print(
    direction_v3_model_panel[
        "minute_features_available"
    ].value_counts(dropna=False)
)

minute_features_available
1    300289
0       986
Name: count, dtype: int64


In [238]:
assert len(direction_v3_model_panel) == len(
    direction_v3_daily_df
)

assert not direction_v3_model_panel.duplicated(
    ["symbol", "pred_date"]
).any()

assert set(V3_MINUTE_FEATURE_COLUMNS).issubset(
    direction_v3_model_panel.columns
)

print(
    "Minute feature missingness:"
)

print(
    direction_v3_model_panel[
        V3_MINUTE_FEATURE_COLUMNS
    ].isna().mean().sort_values(
        ascending=False
    )
)

print("\nFinal V3 panel validation passed.")

Minute feature missingness:
intraday_path_efficiency             0.0033
intraday_realized_volatility         0.0033
intraday_realized_skewness           0.0033
intraday_up_minute_fraction          0.0033
late_session_volatility_share_60m    0.0033
early_session_volatility_share_60m   0.0033
largest_minute_move_share            0.0033
signed_closing_volume_pressure       0.0033
closing_volume_share_60m             0.0033
closing_return_60m                   0.0033
closing_trend_slope                  0.0033
intraday_high_low_range_pct          0.0033
close_location_in_range              0.0033
volume_concentration_hhi             0.0033
minute_bar_coverage                  0.0033
dtype: float64

Final V3 panel validation passed.


In [239]:
direction_v3_model_panel.to_parquet(
    V3_MODEL_PANEL_PATH,
    index=False,
)

print("Saved final V3 panel:")
print(V3_MODEL_PANEL_PATH)
print("Shape:", direction_v3_model_panel.shape)
print(
    "Columns:",
    direction_v3_model_panel.shape[1],
)

Saved final V3 panel:
/Users/kushagr/Desktop/astra-assignment/outputs/processed_data/direction_v3_model_panel.parquet
Shape: (301275, 64)
Columns: 64
